# Automação — Energia Injetada (Geração Distribuída **Celesc / SC**)

Adaptação do pipeline da Copel (PR) para o **"Relatório do Saldo de Energia"** da Celesc Distribuição.
Mesmas etapas (v2: + rateio anual e relatório PDF), mesma arquitetura de planilha (aba-mestre `Demonstrativo` + abas de loja por `SUMIFS`).

| Etapa | O quê | Biblioteca |
|------|-------|-----------|
| **1** | Ler o relatório Celesc (PDF) e extrair, por UC, **todo o histórico** do relatório | `pdfplumber` + `re` |
| **2** | Criar (se não existir) e atualizar `Acompanhamento Energia Injetada SC.xlsx` | `openpyxl` |
| **3** | Gerar o **painel executivo** (PNG/PDF) com status de saldo e recomendações | `matplotlib` |
| **4** | **Rateio anual** (v2): % de cada UC para o próximo ciclo de 12 meses, simulado mês a mês e comparado sob estresse | `numpy` + `scipy` |
| **5** | **Saídas** (v2): abas `Rateio *` no `.xlsx` e **relatório PDF** do demonstrativo | `openpyxl` + `matplotlib` |

---

### O que muda em relação à Copel

| Aspecto | Copel (PR) | **Celesc (SC)** |
|---|---|---|
| Números | decimais BR (`2,32`, `1.234,5`) | **inteiros com ponto de milhar** (`10.704` = 10704). Não há vírgula na tabela |
| Colunas | pares de sub-colunas PT/FP com ordem invertida | **uma coluna por métrica**, linha por posto tarifário |
| Período | `MM/AAAA` | `AAAA/MM` |
| UC | `UC Conces` no topo | `Unidade Consumidora:` no topo (UC ocupa 2 páginas) |
| Percentual | coluna da tabela | **rodapé** da última página da UC: `55625239 (38,77%)` |
| Geradora | tipo no cabeçalho | tabela tem as colunas extras `Excedentes distribuídos` e `Energia alocada na geradora` |

### Mapeamento coluna do PDF → coluna do `Demonstrativo`

| Coluna do relatório Celesc | Coluna no `Demonstrativo` |
|---|---|
| Consumo no período | `consumo` |
| Saldo períodos anteriores | `saldo_anterior_fp` |
| Energia injetada no período | `cred_receb_fp` *(nas beneficiárias = crédito recebido da geradora, calculado pelo balanço do saldo — `CREDITO_PELO_BALANCO`; exibida como "Energia Injetada")* |
| Participação no rodapé (soma das geradoras ≠ `9999999999`) | `percentual` |
| Saldo Final | `saldo_final_fp` |

Validado nos dados: nas beneficiárias, `Energia injetada no período ≈ Excedentes distribuídos da geradora × participação`
(ex.: UC 6115861, 08/2026: 1.551 × 38,77% = 601 ✓). As colunas **A–I** são idênticas às da planilha PR (as fórmulas
são as mesmas); as colunas **J–R** guardam os campos extras da Celesc para auditoria.

> **Padrão da planilha (v3):** **mesmo padrão da planilha PR** — Arial 10, cabeçalhos cinza, abas de loja `31 - BARREIROS - SC` com calendário até dezembro, Resumo Último Mês e Resumo Anual no layout PR, Demonstrativo/Resumos laranja e aba azul `Unidades Consumidoras`. O **nome da loja no Demonstrativo** é o da aba `Unidades Consumidoras`.

## 1. Configuração e logging
Caminhos, **mapa UC → loja**, opções de execução e regras de negócio. Ajuste aqui sem mexer no restante.

> **Nome das lojas (v3):** o nome de cada UC no `Demonstrativo` vem da aba **`Unidades Consumidoras`** da
> planilha (colunas `uc_conces | nome_loja`, ex.: `31 - BARREIROS-SC - 10843693`). Para renomear, edite essa aba e
> rode de novo — com `RENOMEAR_LOJAS_EXISTENTES=True` as linhas já gravadas, as abas de loja (`B2`) e os resumos
> acompanham. `MAPA_UC_LOJA_PADRAO` só é usado se a aba não existir.

In [ ]:
from __future__ import annotations
import logging, sys
from pathlib import Path

# ---------------------------------------------------------------- CAMINHOS
PDF_PATH   = Path("demonstativos") / "[Celesc] Relatório de saldo de Geraçao Distribuída.pdf"   # entrada: relatório Celesc
XLSX_PATH  = Path("Acompanhamento Energia Injetada SC.xlsx")     # planilha SC (criada se não existir)
OUT_DIR    = Path("saida"); OUT_DIR.mkdir(exist_ok=True)
XLSX_OUT   = OUT_DIR / "Acompanhamento Energia Injetada SC.xlsx"
PAINEL_OUT = OUT_DIR / "painel_energia_injetada_sc.pdf"             # sem extensão (gera .png e .pdf)
RELATORIO_OUT = OUT_DIR / "relatorio_energia_injetada_sc.pdf"   # (v2) relatório completo do demonstrativo

# ---------------------------------------------------------------- MAPA UC -> LOJA
# O nome de cada UC vem da aba "Unidades Consumidoras" da planilha (colunas uc_conces | nome_loja) — é o
# nome gravado no Demonstrativo. O dicionário abaixo é só a RESERVA, usada se a aba não existir.
ABA_UCS = "Unidades Consumidoras"
# (v4) A chave da UC é o nº da aba "Unidades Consumidoras" (ex.: 889.823.011-49 -> 88982301149).
MAPA_UC_LOJA_PADRAO: dict[int, str] = {
    365180101141: "30 - CAMPECHE-SC",
    88982301149:  "31 - BARREIROS-SC",
    49815101103:  "32 - SANTA MÔNICA-SC",
    378171201150: "34 - PALHOÇA-SC",
    402816101174: "44 - BIGUAÇU",
}
MAPA_UC_LOJA: dict[int, str] = dict(MAPA_UC_LOJA_PADRAO)   # atualizado com a aba de UCs ao carregar a planilha
# O relatório de saldo da Celesc identifica a UC por OUTRO número (ex.: 10843693). Tabela de
# equivalência nº do relatório -> UC do cadastro. Loja nova: acrescente o par aqui (o notebook avisa
# no log qual nº do relatório não tem equivalência). Se o relatório já trouxer a UC do cadastro, ela é
# reconhecida direto, sem precisar desta tabela.
UC_RELATORIO_PARA_CADASTRO: dict[int, int] = {
    55625239: 365180101141,    # 30 - CAMPECHE (geradora)
    10843693: 88982301149,     # 31 - BARREIROS
    6115861:  49815101103,     # 32 - SANTA MÔNICA
    57001852: 378171201150,    # 34 - PALHOÇA
    59078675: 402816101174,    # 44 - BIGUAÇU
}
UCS_GERADORAS = {365180101141}  # UCs geradoras (completado com as que o relatório marcar como geradora)
SUFIXO_ABA = " - SC"           # nome das abas de loja: "31 - BARREIROS - SC"
UC_PLACEHOLDER = 9999999999          # UC fictícia que a Celesc lista no rodapé com 0,00% — ignorada

# ---------------------------------------------------------------- ESTRUTURA DA ABA-MESTRE
ABA_MESTRE  = "Demonstrativo"
# A..I = IDÊNTICAS à planilha PR (as fórmulas SUMIFS dependem disso). J..R = extras Celesc.
COLS_MESTRE = ["nome_loja", "uc_conces", "consumo", "saldo_anterior_fp",
               "cred_receb_fp", "percentual", "saldo_final_fp", "arquivo", "data_referencia",
               "cliente", "posto", "consumo_faturado", "energia_excedente",
               "saldo_apos_compensacao", "excedentes_distribuidos", "energia_alocada_geradora",
               "tipo", "atualizado_em"]
COL_IDX = {c: i + 1 for i, c in enumerate(COLS_MESTRE)}
COLS_NUMERICAS = ["consumo", "saldo_anterior_fp", "cred_receb_fp", "percentual", "saldo_final_fp",
                  "consumo_faturado", "energia_excedente", "saldo_apos_compensacao",
                  "excedentes_distribuidos", "energia_alocada_geradora"]
ABA_RESUMO_MES = "Resumo Ultimo Mês"
PREFIXO_RESUMO_ANUAL = "Resumo Anual"      # vira "Resumo Anual (AAAA)"
TAG_GERADORA = "GERADORA"

# ---------------------------------------------------------------- OPÇÕES DE EXECUÇÃO
APENAS_ULTIMO_MES   = False  # Celesc traz ~13 meses por UC; ler TUDO permite corrigir meses que chegam atrasados
SOBRESCREVER_MESES  = True   # se (UC, mês) já existe e o PDF novo traz valores diferentes, ATUALIZA a linha
RENOMEAR_LOJAS_EXISTENTES = True  # aplica o MAPA_UC_LOJA também às linhas já gravadas no Demonstrativo
ATUALIZAR_ABAS_LOJA = True   # criar/reconciliar abas por loja
MANTER_MESES_EXISTENTES = True  # mantém os meses já montados nas abas de loja
MESES_ATE_DEZEMBRO = True       # abas de loja com calendário até dezembro do último ano (padrão PR)
TOLERANCIA_KWH      = 2      # tolerância (kWh) das checagens de consistência (arredondamento da Celesc)
# (v4) Crédito recebido pelas beneficiárias calculado pelo BALANÇO do saldo:
#   recebido = saldo final − saldo anterior + (consumo − consumo faturado)
# A coluna "Energia injetada no período" do relatório Celesc vem 0 em várias UCs (ex.: Barreiros) mesmo
# com o crédito lançado — a Celesc abate direto no consumo faturado/saldo. False = usa a coluna do PDF.
CREDITO_PELO_BALANCO = True

# ---------------------------------------------------------------- REGRAS DE NEGÓCIO (Etapa 3)
MES_REFERENCIA_PAINEL = None   # None = mês mais recente; ou "2026-08" para fixar o painel num mês já fechado
LIM_SUPERAVIT      = 2000
LIM_CONFORTAVEL    = 800
LIM_BAIXO          = 1

# ---- % Ideal: com ESTRATEGIA_RATEIO = "W" (célula 4.0) vem do rateio anual; os parâmetros
#      abaixo são das estratégias legadas C/A
HORIZONTE_MESES    = 12       # meses até a próxima revisão do rateio: 12 (1x/ano) ou 6 (2x/ano)
CONSIDERAR_SALDO   = True     # True = estratégia C (desconta o saldo acumulado da loja)
                              # False = estratégia A (uniforme, proporcional só ao consumo médio)
JANELA_MEDIA_MESES = 12       # nº de meses mais recentes na média de consumo (None = todo o histórico)
MESES_BUFFER       = 0.5      # margem de segurança: meses extras de consumo na necessidade (estratégia C)
PISO_PCT           = 0.0      # % mínima por filial (0 = sem piso); o excedente é renormalizado

EMPRESA = "Comercio de Tintas Material Eletrico e Hidraulico Verginia Ltda"
UF      = "Santa Catarina"

# ---------------------------------------------------------------- LOGGING
logger = logging.getLogger("energia_sc")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
logger.addHandler(_h)
logger.info("Configuração carregada. PDF=%s | XLSX=%s", PDF_PATH.name, XLSX_PATH.name)

## 2. Dependências
`pip install pdfplumber openpyxl pandas matplotlib numpy scipy`

In [ ]:
import io
import re
import unicodedata
import datetime as dt
import statistics
from collections import defaultdict
from copy import copy

import pandas as pd

FALHAS = []
try:
    import pdfplumber
except Exception as e:                      # noqa: BLE001
    FALHAS.append(f"pdfplumber ({e})")
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
except Exception as e:                      # noqa: BLE001
    FALHAS.append(f"openpyxl ({e})")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
except Exception as e:                      # noqa: BLE001
    FALHAS.append(f"matplotlib ({e})")

if FALHAS:
    logger.error("Dependências ausentes: %s", "; ".join(FALHAS))
else:
    logger.info("Todas as dependências principais disponíveis.")

## Etapa 1 — Extração do relatório Celesc

1. **Cabeçalho** de cada página: `Unidade Consumidora` e `Cliente` via *regex*.
2. **Tabela** via `extract_table()` — o PDF da Celesc tem grade desenhada e camada de texto, então a leitura
   é direta. As colunas são localizadas **pelo rótulo** (normalizado, sem acento/quebra de linha), o que
   tolera a diferença entre a tabela da geradora (11 colunas) e a das beneficiárias (9 colunas).
3. **Rodapé**: a participação percentual vem na **última página de cada UC**. Somamos as participações de
   todas as geradoras listadas, **exceto** a UC fictícia `9999999999` e a **própria UC** (a geradora aparece
   listando a si mesma com 100%/0% — não é crédito recebido).
4. Uma UC pode ocupar várias páginas: as linhas são **agrupadas por UC** e, se houver mais de um posto
   tarifário no mesmo mês (grupo A: PT/FP), **somadas por mês**.

In [ ]:
def limpar_numero(txt) -> float:
    """Converte texto -> float no padrão do relatório Celesc.

    A Celesc imprime **kWh inteiros com ponto de milhar** e nunca usa vírgula na tabela:
      '10.704' -> 10704   '1.283' -> 1283   '777' -> 777   '1.234.567' -> 1234567
    Se aparecer vírgula (ex.: percentual '38,77'), trata como decimal BR.
    ATENÇÃO: é o oposto da regra da Copel — lá '2.32' era decimal. Não reaproveite a função antiga."""
    if txt is None:
        return 0.0
    if isinstance(txt, (int, float)):
        return float(txt)
    s = str(txt).strip().replace("kWh", "").replace("%", "").replace(" ", "")
    if s in ("", "-", "—"):
        return 0.0
    neg = s.startswith("-")
    s = s.lstrip("-")
    if "," in s:                                        # decimal BR: 1.234,56
        s = s.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(\.\d{3})+", s):         # milhar: 10.704 / 1.234.567
        s = s.replace(".", "")
    elif "." in s:                                      # ponto fora do padrão de milhar -> suspeito
        logger.warning("Número em formato inesperado: %r (interpretado como decimal)", txt)
    s = re.sub(r"[^0-9.]", "", s)
    try:
        v = float(s) if s not in ("", ".") else 0.0
    except ValueError:
        return 0.0
    return -v if neg else v


MESES_BR = {1:"jan",2:"fev",3:"mar",4:"abr",5:"mai",6:"jun",
            7:"jul",8:"ago",9:"set",10:"out",11:"nov",12:"dez"}

def parse_referencia(txt) -> dt.datetime | None:
    """'2026/09' (padrão Celesc) -> datetime(2026, 9, 1). Aceita também 'MM/AAAA'."""
    if not txt:
        return None
    s = str(txt).strip()
    m = re.search(r"(\d{4})\s*[/\-]\s*(\d{1,2})\b", s)
    if m:
        ano, mes = int(m.group(1)), int(m.group(2))
    else:
        m = re.search(r"\b(\d{1,2})\s*[/\-]\s*(\d{4})", s)
        if not m:
            return None
        mes, ano = int(m.group(1)), int(m.group(2))
    return dt.datetime(ano, mes, 1) if 1 <= mes <= 12 else None


def _norm(s) -> str:
    """Normaliza rótulo: sem acento, minúsculo, sem quebras/espaços extras."""
    s = unicodedata.normalize("NFKD", str(s or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.replace("\n", " ")).strip().lower()

### 2.1 Leitura de uma página (cabeçalho, tabela e rodapé)

In [ ]:
# rótulo normalizado (prefixo) -> campo interno. A ordem importa: "consumo faturado" antes de "consumo".
ROTULOS = [
    ("periodo",                 "periodo"),
    ("posto",                   "posto"),
    ("consumo faturado",        "consumo_faturado"),
    ("consumo no periodo",      "consumo"),
    ("energia injetada",        "energia_injetada"),
    ("saldo periodos",          "saldo_anterior"),
    ("energia excedente",       "energia_excedente"),
    ("saldo apos",              "saldo_apos_compensacao"),
    ("excedentes distribuido",  "excedentes_distribuidos"),
    ("energia alocada",         "energia_alocada_geradora"),
    ("saldo final",             "saldo_final"),
]
CAMPOS_NUM = ["consumo", "energia_injetada", "saldo_anterior", "consumo_faturado", "energia_excedente",
              "saldo_apos_compensacao", "excedentes_distribuidos", "energia_alocada_geradora", "saldo_final"]


def ler_cabecalho(texto: str) -> dict:
    info = {"uc": None, "cliente": None}
    m = re.search(r"Unidade\s+Consumidora\s*:?\s*(\d+)", texto, re.I)
    if m:
        info["uc"] = int(m.group(1))
    m = re.search(r"Cliente\s*:?\s*(\d+)", texto, re.I)
    if m:
        info["cliente"] = m.group(1)
    return info


def ler_participacoes(texto: str) -> list[tuple[int, float]]:
    """Rodapé 'UC (s) que injetam energia nesta unidade...': [(uc_geradora, pct), ...].
    O número e o '(xx,xx%)' podem estar quebrados em linhas diferentes — \\s* cobre isso."""
    trecho = texto.split("participação percentual", 1)
    if len(trecho) < 2:
        return []
    return [(int(uc), limpar_numero(p))
            for uc, p in re.findall(r"(\d{5,})\s*\(\s*([\d.,]+)\s*%\s*\)", trecho[1])]


def mapear_colunas(header: list) -> dict:
    """{campo: índice_da_coluna} a partir da linha de cabeçalho da tabela."""
    mapa = {}
    for j, rot in enumerate(header):
        r = _norm(rot)
        for prefixo, campo in ROTULOS:
            if r.startswith(prefixo) and campo not in mapa:
                mapa[campo] = j
                break
    faltando = {"periodo", "consumo", "energia_injetada", "saldo_anterior", "saldo_final"} - set(mapa)
    if faltando:
        raise ValueError(f"Cabeçalho da tabela sem colunas obrigatórias: {sorted(faltando)} | lido: {header}")
    return mapa


def ler_tabela(page) -> tuple[list[dict], bool]:
    """Devolve (linhas, eh_geradora). eh_geradora = tabela tem 'Excedentes distribuídos'."""
    linhas, eh_geradora = [], False
    for tabela in page.extract_tables() or []:
        idx = next((i for i, row in enumerate(tabela)
                    if row and any("periodo" in _norm(c) for c in row)), None)
        if idx is None:
            continue
        mapa = mapear_colunas(tabela[idx])
        eh_geradora = eh_geradora or "excedentes_distribuidos" in mapa
        for row in tabela[idx + 1:]:
            if not row:
                continue
            ref = parse_referencia(row[mapa["periodo"]])
            if ref is None:
                continue
            d = {"referencia": ref,
                 "posto": (row[mapa["posto"]] or "").strip() if "posto" in mapa else ""}
            for campo in CAMPOS_NUM:
                d[campo] = limpar_numero(row[mapa[campo]]) if campo in mapa else 0.0
            linhas.append(d)
    return linhas, eh_geradora

### 2.2 Orquestrador da extração — `extrair_pdf()`

In [ ]:
def uc_int(v) -> int | None:
    """'889.823.011-49' / 88982301149 -> 88982301149 (None se não houver dígitos)."""
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return int(v)
    dig = re.sub(r"\D", "", str(v or ""))
    return int(dig) if dig else None


def uc_cadastro(uc_relatorio: int) -> int:
    """Nº do relatório Celesc -> UC do cadastro (aba Unidades Consumidoras)."""
    uc_relatorio = int(uc_relatorio)
    if uc_relatorio in UC_RELATORIO_PARA_CADASTRO:
        return UC_RELATORIO_PARA_CADASTRO[uc_relatorio]
    if uc_relatorio not in MAPA_UC_LOJA:
        logger.warning("Nº %s do relatório Celesc sem equivalência na aba %r nem em "
                       "UC_RELATORIO_PARA_CADASTRO — acrescente o par para dar nome à loja.",
                       uc_relatorio, ABA_UCS)
    return uc_relatorio


def extrair_pdf(pdf_path: Path = PDF_PATH, apenas_ultimo_mes: bool = APENAS_ULTIMO_MES) -> pd.DataFrame:
    """Percorre o relatório Celesc e devolve um DataFrame no schema de `COLS_MESTRE`."""
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF não encontrado: {pdf_path.resolve()}")

    por_uc = defaultdict(lambda: {"cliente": None, "linhas": [], "geradora": False, "part": []})
    with pdfplumber.open(str(pdf_path)) as pdf:
        logger.info("PDF aberto: %d páginas.", len(pdf.pages))
        if not any((p.extract_text() or "").strip() for p in pdf.pages[:3]):
            raise RuntimeError("PDF sem camada de texto (escaneado). Baixe o relatório novamente no "
                               "portal da Celesc — o original é PDF de texto.")
        for i, page in enumerate(pdf.pages, 1):
            try:
                texto = page.extract_text() or ""
                cab = ler_cabecalho(texto)
                if cab["uc"] is None:
                    logger.warning("Página %d: 'Unidade Consumidora' não encontrada — pulada.", i)
                    continue
                reg = por_uc[cab["uc"]]
                reg["cliente"] = reg["cliente"] or cab["cliente"]
                linhas, ger = ler_tabela(page)
                reg["linhas"] += linhas
                reg["geradora"] |= ger
                reg["part"] += ler_participacoes(texto)
            except Exception as e:                                  # noqa: BLE001
                logger.exception("Falha ao processar página %d: %s", i, e)

    registros = []
    for uc, reg in por_uc.items():
        if not reg["linhas"]:
            logger.warning("UC %s sem linhas de tabela.", uc)
            continue
        # participação recebida de OUTRAS geradoras (exclui placeholder e a própria UC)
        pct = round(sum(p for g, p in reg["part"] if g not in (UC_PLACEHOLDER, uc)), 2)
        if not reg["part"]:
            logger.warning("UC %s: rodapé de participação não encontrado (percentual = 0).", uc)

        # soma por mês (cobre UCs com mais de um posto tarifário no mesmo mês)
        meses = {}
        for ln in reg["linhas"]:
            m = meses.setdefault(ln["referencia"], {c: 0.0 for c in CAMPOS_NUM} | {"postos": set()})
            for c in CAMPOS_NUM:
                m[c] += ln[c]
            if ln["posto"]:
                m["postos"].add(ln["posto"])
        refs = sorted(meses)
        if apenas_ultimo_mes:
            refs = refs[-1:]
        tipo = "geradora" if reg["geradora"] else "beneficiaria"
        uc_cad = uc_cadastro(uc)                                 # (v4) nº do relatório -> UC do cadastro
        if reg["geradora"]:
            UCS_GERADORAS.add(uc_cad)
        n_dif = 0
        for ref in refs:
            m = meses[ref]
            cred = m["energia_injetada"]
            if CREDITO_PELO_BALANCO and not reg["geradora"]:
                # crédito recebido = saldo final − saldo anterior + crédito usado (consumo − faturado)
                bal = max(0.0, round(m["saldo_final"] - m["saldo_anterior"]
                                     + (m["consumo"] - m["consumo_faturado"]), 2))
                n_dif += abs(bal - cred) > TOLERANCIA_KWH
                cred = bal
            registros.append(dict(
                nome_loja=None, uc_conces=uc_cad,
                consumo=m["consumo"],
                saldo_anterior_fp=m["saldo_anterior"],
                cred_receb_fp=cred,
                percentual=pct,
                saldo_final_fp=m["saldo_final"],
                arquivo=pdf_path.name, data_referencia=ref,
                cliente=reg["cliente"], posto="/".join(sorted(m["postos"])),
                consumo_faturado=m["consumo_faturado"],
                energia_excedente=m["energia_excedente"],
                saldo_apos_compensacao=m["saldo_apos_compensacao"],
                excedentes_distribuidos=m["excedentes_distribuidos"],
                energia_alocada_geradora=m["energia_alocada_geradora"],
                tipo=tipo, atualizado_em=None,
            ))
        if n_dif:
            logger.info("UC %s: em %d mês(es) o crédito pelo balanço do saldo difere da coluna "
                        "'Energia injetada' do relatório (usado o balanço).", uc_cad, n_dif)
        logger.info("UC %-10s -> %-12s | %-12s | %2d mês(es) %s–%s | participação %.2f%%",
                    uc, uc_cad, tipo, len(refs), refs[0].strftime("%m/%Y"), refs[-1].strftime("%m/%Y"), pct)

    df = pd.DataFrame(registros, columns=COLS_MESTRE)
    if df.empty:
        logger.warning("Extração vazia.")
        return df
    # UCs com o último mês defasado em relação ao relatório
    ult_geral = df["data_referencia"].max()
    for uc, ult in df.groupby("uc_conces")["data_referencia"].max().items():
        if ult < ult_geral:
            logger.warning("UC %s: último mês no relatório é %s (o mais recente do arquivo é %s).",
                           uc, ult.strftime("%m/%Y"), ult_geral.strftime("%m/%Y"))
    logger.info("Extração concluída: %d registro(s) de %d UC(s).", len(df), df["uc_conces"].nunique())
    return df

### 2.3 Checagens de consistência (auditoria da extração)

Duas conferências automáticas, no estilo da validação empírica já feita no PR:

1. **Saldo**: `saldo_final ≈ saldo_após_compensação + energia_excedente − excedentes_distribuídos`.
2. **Rateio**: para cada beneficiária, `crédito recebido ≈ excedentes_distribuídos(geradora) × participação`
   (crédito recebido pelo balanço do saldo quando `CREDITO_PELO_BALANCO = True`; a Celesc às vezes lança
   o crédito um mês depois, conforme a data de leitura da UC).

Divergências acima de `TOLERANCIA_KWH` são listadas. No rateio, meses em que a beneficiária aparece com
**injeção 0 mas a geradora distribuiu** são marcados como *"crédito ainda não lançado"* — normalmente o mês mais
recente, que a Celesc completa no relatório seguinte (é por isso que `SOBRESCREVER_MESES=True`).

In [ ]:
def validar_extracao(df: pd.DataFrame) -> pd.DataFrame:
    """Devolve um DataFrame de ocorrências (vazio = tudo consistente)."""
    ocorr = []
    if df is None or df.empty:
        return pd.DataFrame(ocorr)
    for _, r in df.iterrows():
        esperado = r.saldo_apos_compensacao + r.energia_excedente - r.excedentes_distribuidos
        if abs(esperado - r.saldo_final_fp) > TOLERANCIA_KWH:
            ocorr.append(dict(checagem="saldo", uc=r.uc_conces, mes=r.data_referencia.strftime("%m/%Y"),
                              esperado=esperado, lido=r.saldo_final_fp,
                              obs="saldo final ≠ saldo após + excedente − distribuídos"))
    ger = df[df["tipo"] == "geradora"].groupby("data_referencia")["excedentes_distribuidos"].sum()
    ult_uc = df.groupby("uc_conces")["data_referencia"].max()
    for _, r in df[df["tipo"] == "beneficiaria"].iterrows():
        dist = ger.get(r.data_referencia)
        if dist is None:
            continue
        esperado = round(dist * r.percentual / 100)
        if abs(esperado - r.cred_receb_fp) > TOLERANCIA_KWH:
            if r.cred_receb_fp == 0 and r.data_referencia == ult_uc[r.uc_conces]:
                obs = "mês mais recente: crédito ainda não lançado na beneficiária"
            elif r.cred_receb_fp == 0:
                obs = "injeção 0 com a geradora distribuindo — conferir"
            else:
                obs = "diferente do rateio vigente (participação pode ter mudado no período)"
            ocorr.append(dict(checagem="rateio", uc=r.uc_conces, mes=r.data_referencia.strftime("%m/%Y"),
                              esperado=esperado, lido=r.cred_receb_fp, obs=obs))
    out = pd.DataFrame(ocorr)
    if out.empty:
        logger.info("Validação: nenhuma divergência.")
    else:
        logger.warning("Validação: %d ocorrência(s) — veja a tabela retornada.", len(out))
    return out

## Etapa 2 — Planilha SC (`openpyxl`)

Diferente do PR, a planilha SC **ainda não existe**. `garantir_estrutura()` cria (ou completa) a mesma arquitetura:

```
Demonstrativo (A–I iguais ao PR, J–R extras Celesc)
 ├──► uma aba por loja  (A1 título | A2 "Loja (ref.):" B2 nome | linha 3 cabeçalho | meses | TOTAL)
 ├──► "Resumo Ultimo Mês"      → SUMIFS com P1 = =MAX(Demonstrativo!I:I)
 └──► "Resumo Anual (AAAA)"    → SUMIFS do ano de P1
```

Fluxo a cada execução:
1. Carregar a planilha SC (ou criar do zero).
2. **Upsert** no `Demonstrativo`: chave `(uc, AAAA-MM)`. Linha nova → insere; linha existente com valores
   diferentes → **atualiza** (se `SOBRESCREVER_MESES`). Rodar 2× o mesmo PDF não duplica nada.
3. Criar abas para lojas novas e **reconciliar** todas as abas de loja (`sincronizar_abas_loja`, igual ao PR).
4. Reconstruir os resumos com a lista atual de lojas.

In [ ]:
# ---- padrão visual das planilhas PR/SC: cabeçalhos cinza CCCCCC com texto preto em negrito
_FILL_H  = PatternFill("solid", fgColor="CCCCCC")
_FILL_H2 = _FILL_H
_FILL_T  = _FILL_H                                   # linha TOTAL
_FONT_H  = Font(bold=True, color="000000")
_BORDA   = Border(bottom=Side(style="thin", color="C9D4D4"))
FMT_KWH, FMT_MES, FMT_PCT = "#,##0", "mmm/yyyy", "0.0%"
COR_ABA_BASE, COR_ABA_UCS = "FF9900", "0000FF"       # Demonstrativo/Resumos laranja; Unidades Consumidoras azul
RENOMEADOS: dict = {}                                # nome antigo -> nome da aba de UCs (preenchido no upsert)


def nome_loja(uc: int) -> str:
    nome = MAPA_UC_LOJA.get(int(uc), f"UC {int(uc)}")
    if int(uc) in UCS_GERADORAS and TAG_GERADORA.upper() not in nome.upper():
        nome = f"{nome} ({TAG_GERADORA})"                  # padrão "80 - INDUSTRIA (GERADORA)"
    return nome


def titulo_loja(loja: str) -> str:
    """'31 - BARREIROS-SC - 10843693' -> '31 - BARREIROS - SC' (título e nome da aba da loja)."""
    base = re.sub(r"\s*\(%s\)\s*$" % TAG_GERADORA, "", str(loja), flags=re.I).strip()   # tira "(GERADORA)"
    base = re.sub(r"\s+-\s*\d{5,}\s*$", "", base).strip()            # tira a UC do fim
    if base.upper().startswith(("SC ", "UC ")):
        return base
    base = re.sub(r"\s*-\s*SC\s*$", "", base, flags=re.I).strip()        # normaliza o sufixo
    return base + SUFIXO_ABA if SUFIXO_ABA else base


def _rotulo_curto(loja: str) -> str:
    """Nome curto para o rodapé do painel: '31 - BARREIROS-SC - 10843693' -> 'BARREIROS-SC'."""
    return re.sub(r"\s+-\s*\d{5,}\s*$", "", str(loja)).split(" - ")[-1]


def _nome_aba(loja: str) -> str:
    return re.sub(r"[\[\]\*\?/\\:]", "-", titulo_loja(loja))[:31]


def _nome_aba_unico(wb, loja: str) -> str:
    base, k, nome = _nome_aba(loja), 2, _nome_aba(loja)
    while nome in wb.sheetnames:
        suf = f" ({k})"; nome = base[:31 - len(suf)] + suf; k += 1
    return nome


def carregar_mapa_ucs(wb) -> dict:
    """Atualiza MAPA_UC_LOJA com a aba 'Unidades Consumidoras' (uc_conces | nome_loja)."""
    mapa = {}
    if ABA_UCS in wb.sheetnames:
        for uc, nome in wb[ABA_UCS].iter_rows(min_row=2, max_col=2, values_only=True):
            uc = uc_int(uc)                                    # aceita 889.823.011-49
            if uc and nome:
                mapa[uc] = str(nome).strip()
        logger.info("Aba %r: %d UC(s) mapeadas.", ABA_UCS, len(mapa))
    else:
        logger.warning("Aba %r não encontrada — usando MAPA_UC_LOJA_PADRAO.", ABA_UCS)
    MAPA_UC_LOJA.clear()
    MAPA_UC_LOJA.update({**MAPA_UC_LOJA_PADRAO, **mapa})
    return MAPA_UC_LOJA


def garantir_aba_ucs(wb):
    """Cria a aba 'Unidades Consumidoras' (se faltar) e acrescenta UCs do Demonstrativo que não estão nela."""
    if ABA_UCS not in wb.sheetnames:
        ws = wb.create_sheet(ABA_UCS)
        ws["A1"], ws["B1"] = "uc_conces", "nome_loja"
        ws["A1"].font = ws["B1"].font = Font(bold=True)
        ws.column_dimensions["A"].width, ws.column_dimensions["B"].width = 12, 40.4
    ws = wb[ABA_UCS]
    conhecidas = set()
    for (uc,) in ws.iter_rows(min_row=2, max_col=1, values_only=True):
        if uc_int(uc):
            conhecidas.add(uc_int(uc))
    dem = wb[ABA_MESTRE]
    ucs = {int(dem.cell(r, COL_IDX["uc_conces"]).value) for r in range(2, dem.max_row + 1)
           if dem.cell(r, COL_IDX["uc_conces"]).value is not None}
    prox = max([r for r in range(1, ws.max_row + 1) if ws.cell(r, 1).value is not None] or [1]) + 1
    for uc in sorted(ucs - conhecidas):
        ws.cell(prox, 1, uc); ws.cell(prox, 2, nome_loja(uc)); prox += 1
        logger.info("Aba %r: UC %s acrescentada como %r.", ABA_UCS, uc, nome_loja(uc))
    ws.sheet_properties.tabColor = COR_ABA_UCS


# ---- tipografia e alinhamentos da planilha PR (Arial 10, células centralizadas)
_FONTE = "Arial"
_F10, _F10B = Font(name=_FONTE, size=10), Font(name=_FONTE, size=10, bold=True)
_F10H = Font(name=_FONTE, size=10, bold=True, color="000000")
_F13H = Font(name=_FONTE, size=13, bold=True, color="000000")
_FIN = Side(style="thin")
_GRADE = Border(left=_FIN, right=_FIN, top=_FIN, bottom=_FIN)
_SEM_BORDA = Border()
_CENTRO = Alignment(horizontal="center", vertical="center")
_BAIXO = Alignment(vertical="bottom")


def _criar_demonstrativo(wb):
    ws = wb.create_sheet(ABA_MESTRE)
    for j, c in enumerate(COLS_MESTRE, 1):
        ws.cell(1, j, c)
    _estilo_demonstrativo(ws)
    return ws


def _estilo_demonstrativo(ws):
    """Demonstrativo como na planilha PR: cabeçalho cinza com grade fina, loja em negrito, data mm/yyyy."""
    ncol = len(COLS_MESTRE)
    for j in range(1, ncol + 1):
        c = ws.cell(1, j); c.font = _F10H; c.fill = _FILL_H; c.border = _GRADE
        c.alignment = Alignment(horizontal=None if j == 1 else ("left" if j == 2 else "center"), vertical="bottom")
    for r in range(2, ws.max_row + 1):
        if ws.cell(r, 1).value is None:
            continue
        for j in range(1, ncol + 1):
            c = ws.cell(r, j)
            c.font = _F10B if j == 1 else _F10
            c.alignment = Alignment(horizontal=None if j == 1 else ("left" if j == 2 else "center"), vertical="bottom")
        ws.cell(r, COL_IDX["data_referencia"]).number_format = "mm/yyyy"
    for L, w in {"A": 30, "B": 12, "G": 26.75, "H": 28, "I": 22.38}.items():
        ws.column_dimensions[L].width = w
    ws.freeze_panes = None


def _criar_aba_loja(wb, loja: str):
    """Mesmo layout das abas de loja da planilha PR, com 1 linha-modelo + TOTAL."""
    ws = wb.create_sheet(_nome_aba_unico(wb, loja), index=0)
    ws["A1"] = titulo_loja(loja)
    ws.merge_cells("A1:F1")
    ws["A2"] = "Loja (ref.):"; ws["B2"] = loja; ws.merge_cells("B2:F2")
    for j, h in enumerate(["Mês/Ano", "Consumo (kWh)", "Saldo Anterior (kWh)",
                           "Energia Injetada (kWh)", "Saldo Final (kWh)", "Diferença %"], 1):
        ws.cell(3, j, h)
    ws["A5"] = "TOTAL"
    _estilo_aba_loja(ws)
    logger.info("Aba de loja criada: %r", ws.title)
    return ws


def _estilo_aba_loja(ws):
    """Aba de loja como na planilha PR: título cinza 13, cabeçalho cinza, linhas Arial 10 centralizadas."""
    ws["A1"].font = _F13H; ws["A1"].fill = _FILL_H; ws["A1"].alignment = _CENTRO
    ws["A2"].font = _F10B; ws["A2"].alignment = Alignment(horizontal="left", vertical="center")
    ws["B2"].font = _F10; ws["B2"].alignment = _BAIXO
    fmts = [FMT_MES, FMT_KWH, FMT_KWH, FMT_KWH, FMT_KWH, "0.00%"]
    for r in range(3, ws.max_row + 1):
        total = str(ws.cell(r, 1).value).strip().upper() == "TOTAL"
        if r > 3 and ws.cell(r, 1).value is None and not total:
            continue
        for j in range(1, 7):
            c = ws.cell(r, j)
            c.alignment = _CENTRO; c.border = _SEM_BORDA
            if r == 3 or total:
                c.font = _F10H; c.fill = _FILL_H
            else:
                c.font = _F10; c.fill = PatternFill(fill_type=None)
            if r > 3:
                c.number_format = ("0.0%" if j == 6 else FMT_KWH) if total else fmts[j - 1]
            if total and j == 1:
                c.number_format = "General"
        ws.row_dimensions[r].height = 15
    ws.row_dimensions[1].height = 21.75
    ws.row_dimensions[2].height = 15
    for L, w in zip("ABCDEF", [12, 15, 20, 22, 18, 13]):
        ws.column_dimensions[L].width = w
    ws.freeze_panes = None


def carregar_ou_criar_workbook(xlsx_path: Path = XLSX_PATH):
    if not xlsx_path.exists() and XLSX_OUT.exists():
        # continuidade: sem planilha-base, reaproveita a última saída (preserva o histórico acumulado)
        logger.info("%s não existe — continuando a partir de %s.", xlsx_path.name, XLSX_OUT.name)
        xlsx_path = XLSX_OUT
    if xlsx_path.exists():
        wb = openpyxl.load_workbook(io.BytesIO(Path(xlsx_path).read_bytes()), data_only=False)
        logger.info("Workbook carregado: %s (%d abas).", xlsx_path.name, len(wb.sheetnames))
    else:
        wb = openpyxl.Workbook(); wb.remove(wb.active)
        logger.info("Workbook %s não existe — criando estrutura nova.", xlsx_path.name)
    if ABA_MESTRE not in wb.sheetnames:
        _criar_demonstrativo(wb)
    migrar_ucs_demonstrativo(wb)
    carregar_mapa_ucs(wb)
    return wb


def migrar_ucs_demonstrativo(wb) -> int:
    """(v4) Linhas gravadas com o nº do relatório Celesc passam a usar a UC do cadastro."""
    ws, n = wb[ABA_MESTRE], 0
    for r in range(2, ws.max_row + 1):
        c = ws.cell(r, COL_IDX["uc_conces"])
        uc = uc_int(c.value)
        if uc is None:
            continue
        if uc in UC_RELATORIO_PARA_CADASTRO:
            c.value = UC_RELATORIO_PARA_CADASTRO[uc]
            n += 1
        c.number_format = "0"
    if n:
        logger.info("Demonstrativo: %d linha(s) migrada(s) do nº do relatório para a UC do cadastro.", n)
    return n


def _lojas_no_mestre(wb) -> list[str]:
    ws = wb[ABA_MESTRE]
    return sorted({str(v) for v in (ws.cell(r, 1).value for r in range(2, ws.max_row + 1)) if v})


def _abas_loja(wb) -> list:
    return [ws for ws in wb.worksheets if ws.cell(2, 1).value == "Loja (ref.):"]


def garantir_abas_loja(wb):
    # abas que ainda usam o nome antigo (ex.: "SC - UC 10843693") passam a usar o da aba de UCs
    for ws in _abas_loja(wb):
        antigo = ws.cell(2, 2).value
        if antigo in RENOMEADOS:
            novo = RENOMEADOS[antigo]
            ws.cell(2, 2, novo)
            if ws["A1"].value in (None, antigo) or str(ws["A1"].value).strip() == str(antigo).strip():
                ws["A1"] = titulo_loja(novo)
            if ws.title == re.sub(r"[\[\]\*\?/\\:]", "-", str(antigo))[:31]:
                ws.title = _nome_aba_unico(wb, novo)
            logger.info("Aba %r: loja de referência %r → %r.", ws.title, antigo, novo)
    # padrão PR: título (A1) = nome da aba = nome padronizado da loja ("44 - BIGUAÇU - SC")
    for ws in _abas_loja(wb):
        loja = ws.cell(2, 2).value
        if not loja:
            continue
        ws["A1"] = titulo_loja(loja)
        alvo = _nome_aba(loja)
        if ws.title != alvo and alvo not in wb.sheetnames:
            logger.info("Aba %r renomeada para %r.", ws.title, alvo)
            ws.title = alvo
    existentes = {ws.cell(2, 2).value for ws in _abas_loja(wb)}
    atuais = _lojas_no_mestre(wb)
    for loja in atuais:
        if loja not in existentes:
            _criar_aba_loja(wb, loja)
    for orfa in sorted(existentes - set(atuais) - {None}):
        logger.warning("Aba de loja %r não tem mais dados no Demonstrativo (loja renomeada?). "
                       "Pode ser apagada manualmente.", orfa)
    reordenar_abas(wb)


def reordenar_abas(wb):
    """Abas de loja (por nome), Demonstrativo, resumos, Unidades Consumidoras e, por último, as demais."""
    lojas = sorted(_abas_loja(wb), key=lambda w: w.title)
    base = [wb[n] for n in [ABA_MESTRE, ABA_RESUMO_MES] if n in wb.sheetnames]
    base += [ws for ws in wb.worksheets if ws.title.startswith(PREFIXO_RESUMO_ANUAL)]
    base += [wb[ABA_UCS]] if ABA_UCS in wb.sheetnames else []
    outras = [ws for ws in wb.worksheets if ws not in lojas and ws not in base]
    wb._sheets = lojas + base + outras


def aplicar_estilo_padrao(wb):
    """Mesmo padrão visual da planilha PR (Arial 10, cinza CCCCCC, abas laranja/azul), mesmo partindo de
    uma planilha antiga ou da planilha-modelo SC."""
    normal = wb._named_styles["Normal"]
    normal.font = Font(name=_FONTE, size=10)                   # fonte padrão da pasta = Arial 10 (como no PR)
    for ws in _abas_loja(wb):
        _estilo_aba_loja(ws)
    _estilo_demonstrativo(wb[ABA_MESTRE])
    if ABA_UCS in wb.sheetnames:
        wu = wb[ABA_UCS]
        for r in range(1, wu.max_row + 1):
            for j in (1, 2):
                c = wu.cell(r, j)
                c.font = Font(name=_FONTE, bold=(r == 1))
                c.alignment = Alignment(horizontal="left" if j == 1 else "center", vertical="bottom")
        wu.column_dimensions["A"].width, wu.column_dimensions["B"].width = 9.63, 32.25
    for ws in wb.worksheets:
        if ws.title in (ABA_MESTRE, ABA_RESUMO_MES) or ws.title.startswith(PREFIXO_RESUMO_ANUAL):
            ws.sheet_properties.tabColor = COR_ABA_BASE
    if ABA_UCS in wb.sheetnames:
        wb[ABA_UCS].sheet_properties.tabColor = COR_ABA_UCS

### 3.1 Upsert na aba-mestre

In [ ]:
def _ultima_linha(ws, col=1) -> int:
    last = 1
    for r in range(1, ws.max_row + 1):
        if ws.cell(r, col).value is not None:
            last = r
    return last


def upsert_demonstrativo(wb, df: pd.DataFrame) -> dict:
    """Insere linhas novas e (opcionalmente) atualiza as existentes cujo valor mudou.
    Devolve {'inseridas': [...], 'atualizadas': [...]} com (loja, 'MM/AAAA')."""
    ws = wb[ABA_MESTRE]
    indice = {}                                   # (uc, 'AAAA-MM') -> nº da linha
    for r in range(2, ws.max_row + 1):
        uc, ref = ws.cell(r, COL_IDX["uc_conces"]).value, ws.cell(r, COL_IDX["data_referencia"]).value
        if uc is not None and hasattr(ref, "strftime"):
            indice[(int(uc), ref.strftime("%Y-%m"))] = r
            if RENOMEAR_LOJAS_EXISTENTES and int(uc) in MAPA_UC_LOJA:
                antigo, novo = ws.cell(r, 1).value, nome_loja(int(uc))
                if antigo is not None and antigo != novo:
                    RENOMEADOS[antigo] = novo                       # propaga às abas de loja
                ws.cell(r, 1, novo)
    prox = _ultima_linha(ws) + 1
    agora = dt.datetime.now().replace(microsecond=0)
    res = {"inseridas": [], "atualizadas": []}

    for _, row in df.iterrows():
        uc, ref = int(row["uc_conces"]), row["data_referencia"]
        vals = {c: row[c] for c in COLS_MESTRE}
        vals["nome_loja"] = nome_loja(uc)
        vals["atualizado_em"] = agora
        chave = (uc, ref.strftime("%Y-%m"))
        rotulo = (vals["nome_loja"], ref.strftime("%m/%Y"))
        if chave in indice:
            r = indice[chave]
            mudou = [c for c in COLS_NUMERICAS
                     if abs(float(ws.cell(r, COL_IDX[c]).value or 0) - float(vals[c] or 0)) > 1e-9]
            if not mudou or not SOBRESCREVER_MESES:
                if mudou:
                    logger.warning("%s %s: PDF difere (%s) mas SOBRESCREVER_MESES=False.", *rotulo, mudou)
                continue
            logger.info("Atualizada %s %s — campos: %s", *rotulo, ", ".join(mudou))
            res["atualizadas"].append(rotulo)
        else:
            r = prox; prox += 1; indice[chave] = r
            res["inseridas"].append(rotulo)
        for c, v in vals.items():
            cell = ws.cell(r, COL_IDX[c], v)
            if c == "data_referencia":
                cell.number_format = "dd/mm/yyyy"
            elif c == "atualizado_em":
                cell.number_format = "dd/mm/yyyy hh:mm"
            elif c == "uc_conces":
                cell.number_format = "0"
    logger.info("Demonstrativo: %d inserida(s), %d atualizada(s).",
                len(res["inseridas"]), len(res["atualizadas"]))
    return res

### 3.2 Reconciliação das abas de loja (mesma lógica do PR)
Garante que cada aba de loja contenha exatamente os meses do `Demonstrativo` — recria meses apagados à mão.

In [ ]:
def _achar_linha_total(ws) -> int | None:
    for r in range(1, ws.max_row + 1):
        v = ws.cell(r, 1).value
        if isinstance(v, str) and v.strip().upper() == "TOTAL":
            return r
    return None


def _sumifs(col_dem: str, ln: int) -> str:
    """Fórmula SUMIFS idêntica ao padrão da planilha, para a coluna `col_dem` do Demonstrativo."""
    return (f'=SUMIFS(Demonstrativo!{col_dem}:{col_dem}, Demonstrativo!A:A, $B$2, '
            f'Demonstrativo!I:I, ">="&$A{ln}, Demonstrativo!I:I, "<="&EOMONTH($A{ln},0))')


PRIMEIRA_LINHA_DADOS = 4   # 1ª linha de dados nas abas de loja (linhas 1-3 = cabeçalho)


def _meses_no_mestre(dem, nome_loja: str) -> list:
    """(ano, mês) distintos presentes na aba-mestre para a loja (casando por nome, col A)."""
    ms = set()
    for r in range(2, dem.max_row + 1):
        if dem.cell(r, 1).value == nome_loja:
            ref = dem.cell(r, COL_IDX["data_referencia"]).value
            if hasattr(ref, "year"):
                ms.add((ref.year, ref.month))
    return sorted(ms)


def sincronizar_abas_loja(wb, apenas_datas: set | None = None) -> list:
    """Reconcilia CADA aba de loja com a aba-mestre `Demonstrativo`.

    Diferentemente de uma inserção gatilhada apenas pelos meses recém-lidos do PDF,
    esta função é **declarativa/idempotente**: ela garante que toda aba de loja
    contenha exatamente as linhas de mês que existem no `Demonstrativo` para aquela
    loja. Portanto ela **repara divergências** — por exemplo, um mês apagado
    manualmente da aba individual é recriado, mesmo sem PDF novo.

    O bloco de meses é reescrito em posições FINAIS (o openpyxl NÃO reajusta
    referências de fórmula ao inserir/remover linhas; reescrever no lugar evita isso).
    `apenas_datas` (opcional) restringe o reparo a esses meses (os demais são mantidos).
    Devolve [(aba, [meses_restaurados])].
    """
    if not ATUALIZAR_ABAS_LOJA:
        return []
    dem = wb[ABA_MESTRE]
    especiais = {ABA_MESTRE, ABA_RESUMO_MES}
    relatorio = []
    for nome in wb.sheetnames:
        if nome in especiais or nome.startswith(PREFIXO_RESUMO_ANUAL):
            continue
        ws = wb[nome]
        if ws.cell(2, 1).value != "Loja (ref.):":            # confirma que é aba de loja
            continue
        nome_loja = ws.cell(2, 2).value                       # B2 = critério dos SUMIFS
        meses_alvo = _meses_no_mestre(dem, nome_loja)
        if not meses_alvo:
            continue
        total_row = _achar_linha_total(ws)
        if total_row is None:
            logger.warning("Aba %r: linha TOTAL não encontrada — pulada.", nome)
            continue

        # meses já presentes na aba (para o relatório do que foi restaurado)
        antes = {(v.year, v.month) for v in
                 (ws.cell(r, 1).value for r in range(PRIMEIRA_LINHA_DADOS, total_row))
                 if isinstance(v, dt.datetime)}
        if MANTER_MESES_EXISTENTES:          # mantém os meses já montados na aba (ex.: calendário até dezembro)
            meses_alvo = sorted(set(meses_alvo) | antes)
        if MESES_ATE_DEZEMBRO and meses_alvo:   # calendário completo até dezembro, como nas abas PR
            (a0, m0), a1 = meses_alvo[0], meses_alvo[-1][0]
            meses_alvo = sorted(set(meses_alvo) | {(a, m) for a in range(a0, a1 + 1) for m in range(1, 13)
                                                   if (a, m) >= (a0, m0)})
        if apenas_datas is not None:
            filtro = {(d.year, d.month) for d in apenas_datas}
            meses_alvo = sorted(antes | (set(meses_alvo) & filtro))

        n_alvo = len(meses_alvo)
        # estilo-modelo (1ª linha de dados) para aplicar a linhas novas/uniformizar
        estilo = [copy(ws.cell(PRIMEIRA_LINHA_DADOS, c)._style) for c in range(1, 7)]

        # ajusta a quantidade de linhas físicas do bloco para casar com n_alvo
        linhas_atuais = total_row - PRIMEIRA_LINHA_DADOS
        if n_alvo > linhas_atuais:
            ws.insert_rows(total_row, n_alvo - linhas_atuais)          # abre espaço acima do TOTAL
        elif n_alvo < linhas_atuais:
            ws.delete_rows(PRIMEIRA_LINHA_DADOS, linhas_atuais - n_alvo)
        total_row = PRIMEIRA_LINHA_DADOS + n_alvo                       # TOTAL na nova posição

        # reescreve o bloco em posições finais
        for i, (ano, mes) in enumerate(meses_alvo):
            r = PRIMEIRA_LINHA_DADOS + i
            for c in range(1, 7):
                ws.cell(r, c)._style = copy(estilo[c - 1])
            ws.cell(r, 1, dt.datetime(ano, mes, 1))
            ws.cell(r, 2, _sumifs("C", r))    # Consumo
            ws.cell(r, 3, _sumifs("D", r))    # Saldo Anterior
            ws.cell(r, 4, _sumifs("E", r))    # Energia Injetada (cred_receb_fp)
            ws.cell(r, 5, _sumifs("G", r))    # Saldo Final
            ws.cell(r, 6, f'=IFERROR((D{r}-B{r})/B{r},"")')            # Diferença %

        # TOTAL (SUM cobrindo todo o bloco)
        ult = PRIMEIRA_LINHA_DADOS + n_alvo - 1
        for c, L in [(2, "B"), (3, "C"), (4, "D"), (5, "E")]:
            ws.cell(total_row, c, f"=SUM({L}{PRIMEIRA_LINHA_DADOS}:{L}{ult})")
        ws.cell(total_row, 6, f'=IFERROR((D{total_row}-B{total_row})/B{total_row},"")')

        restaurados = sorted(set(meses_alvo) - antes)
        if restaurados:
            logger.info("Aba %r: %d mês(es) restaurado(s): %s", nome, len(restaurados),
                        ", ".join(f"{m:02d}/{a}" for a, m in restaurados))
            relatorio.append((nome, [f"{m:02d}/{a}" for a, m in restaurados]))
    if not relatorio:
        logger.info("Abas de loja já sincronizadas com o Demonstrativo (nada a reparar).")
    return relatorio

### 3.3 Resumos (reconstruídos a cada execução)

In [ ]:
def _sumifs_mes(col: str, r: int) -> str:
    return (f'=SUMIFS(Demonstrativo!{col}:{col}, Demonstrativo!A:A, A{r}, Demonstrativo!I:I, '
            f'">="&DATE(YEAR($P$1), MONTH($P$1), 1), Demonstrativo!I:I, "<="&EOMONTH($P$1, 0))')


def _cab(ws, linha, titulos, col0=1, nf=None):
    """Cabeçalho no padrão PR: cinza, Arial 10 negrito preto, grade fina."""
    for j, h in enumerate(titulos, col0):
        c = ws.cell(linha, j, h); c.font = _F10H; c.fill = _FILL_H; c.border = _GRADE
        c.alignment = (Alignment(horizontal="center", vertical="center", wrap_text=True) if j == 1
                       else Alignment(vertical="bottom"))
        if nf and j > 1:
            c.number_format = nf


def _loja_col_a(c):
    c.font = Font(name=_FONTE, size=10, bold=True, color="000000"); c.border = _GRADE
    c.alignment = Alignment(vertical="bottom")


def _valor(c, nf=FMT_KWH, centro=True):
    c.font = Font(name=_FONTE, size=10, color="000000"); c.number_format = nf
    c.alignment = Alignment(horizontal="center" if centro else None, vertical="bottom")


def reconstruir_resumos(wb):
    """Recria 'Resumo Ultimo Mês' e 'Resumo Anual (AAAA)' no mesmo layout da planilha PR (idempotente)."""
    lojas = _lojas_no_mestre(wb)
    dem = wb[ABA_MESTRE]
    refs = [v for v in (dem.cell(r, COL_IDX["data_referencia"]).value for r in range(2, dem.max_row + 1))
            if hasattr(v, "year")]
    ano = max(refs).year if refs else dt.date.today().year
    for n in list(wb.sheetnames):
        if n == ABA_RESUMO_MES or n.startswith(PREFIXO_RESUMO_ANUAL):
            del wb[n]

    # ---- Resumo Último Mês (layout PR)
    ws = wb.create_sheet(ABA_RESUMO_MES)
    _cab(ws, 1, ["LOJA", "Energia Consumida Ultimo Mês", "Energia Injetada Ultimo Mês", "Saldo Total (KWh)"], nf=FMT_KWH)
    _cab(ws, 1, ["Mês de Referência"], col0=15)
    ws.cell(1, 15).alignment = Alignment(vertical="bottom")
    ws["P1"] = "=MAX(Demonstrativo!I:I)"
    _valor(ws["P1"], "m/d/yyyy", centro=False)
    for i, loja in enumerate(lojas, 2):
        c = ws.cell(i, 1, loja); _loja_col_a(c)
        c.alignment = Alignment(vertical="center", wrap_text=True)
        for j, col in enumerate("CEG", 2):
            _valor(ws.cell(i, j, _sumifs_mes(col, i)))
    _cab(ws, 1, ["Métrica", "Valor (KWh)"], col0=18)
    ws.cell(1, 18).alignment = Alignment(vertical="bottom")
    metricas = [("Energia Consumida Total", "C"), ("Energia Injetada (CD)", "E")]
    for k, (m, col) in enumerate(metricas, 2):
        c = ws.cell(k, 18, m); c.font = Font(name=_FONTE, size=10, bold=True, color="000000")
        c.alignment = Alignment(vertical="bottom")
        _valor(ws.cell(k, 19, f'=SUMIFS(Demonstrativo!{col}:{col}, Demonstrativo!I:I, P1)'), centro=False)
    for L, w in {"A": 32, "B": 26, "C": 23.25, "D": 15.25, "O": 15.63, "P": 9.5, "R": 21, "S": 10.5}.items():
        ws.column_dimensions[L].width = w
    ws.sheet_properties.tabColor = COR_ABA_BASE

    # ---- Resumo Anual (layout PR): LOJA | Consumida | Injetada | Diferença % + totais no rodapé
    wa = wb.create_sheet(f"{PREFIXO_RESUMO_ANUAL} ({ano})")
    _cab(wa, 1, ["LOJA", f"Energia Consumida Total {ano}", f"Energia Injetada Total {ano}", "Diferença %"], nf=FMT_KWH)
    wa.cell(1, 1).alignment = Alignment(vertical="bottom"); wa.cell(1, 4).number_format = "General"
    ini, fim = f"DATE({ano},1,1)", f"DATE({ano},12,31)"
    for i, loja in enumerate(lojas, 2):
        _loja_col_a(wa.cell(i, 1, loja))
        for j, col in enumerate("CE", 2):
            _valor(wa.cell(i, j, (f'=SUMIFS(Demonstrativo!{col}:{col}, Demonstrativo!A:A, A{i}, '
                                  f'Demonstrativo!I:I, ">="&{ini}, Demonstrativo!I:I, "<="&{fim})')))
        _valor(wa.cell(i, 4, f'=IFERROR((C{i}-B{i})/B{i},"")'), "0.00%", centro=False)
    ult = len(lojas) + 1
    for k, (rot, col) in enumerate([("Total Injetada ", "C"), ("Total Consumida", "B")]):
        r = ult + 2 + k
        c = wa.cell(r, 1, rot); c.font = Font(name=_FONTE, bold=True); c.border = _GRADE
        c = wa.cell(r, 2, f"=SUM({col}2:{col}{ult})"); c.font = Font(name=_FONTE); c.number_format = FMT_KWH
    for L, w in {"A": 32, "B": 25.13, "C": 22.38, "D": 11.75}.items():
        wa.column_dimensions[L].width = w
    wa.sheet_properties.tabColor = COR_ABA_BASE
    logger.info("Resumos reconstruídos (%d lojas, ano %d).", len(lojas), ano)

### 3.4 Orquestrador da Etapa 2 — `atualizar_excel()`

In [ ]:
def atualizar_excel(df: pd.DataFrame | None = None, xlsx_in: Path = XLSX_PATH,
                    xlsx_out: Path = XLSX_OUT) -> Path:
    """Cria/abre a planilha SC, faz upsert no Demonstrativo, reconcilia abas e recria resumos.
    `df=None` = modo reparo (só reconcilia, sem ler PDF)."""
    wb = carregar_ou_criar_workbook(xlsx_in)
    if df is not None and not df.empty:
        upsert_demonstrativo(wb, df)
    else:
        logger.info("Etapa 2 | sem dados novos — apenas reconciliação.")
    garantir_abas_loja(wb)
    sincronizar_abas_loja(wb)
    reconstruir_resumos(wb)
    garantir_aba_ucs(wb)
    aplicar_estilo_padrao(wb)
    reordenar_abas(wb)
    wb.save(str(xlsx_out))
    logger.info("Planilha salva em: %s  (o Excel recalcula as fórmulas ao abrir)", xlsx_out)
    return xlsx_out


def reparar_planilha(xlsx_in: Path = XLSX_PATH, xlsx_out: Path = XLSX_OUT) -> Path:
    return atualizar_excel(df=None, xlsx_in=xlsx_in, xlsx_out=xlsx_out)

## Etapa 3 — Painel executivo

Igual ao PR (KPIs, tabela por filial com **% Atual / % Ideal / Inj. p/ Ideal (~kWh)**, chips de status e
rodapé de recomendações), com um ajuste para a Celesc:

> **Lojas com mês defasado.** A Celesc às vezes não traz o mês mais recente de uma UC (ex.: UC 10843693 termina
> em 08/2026 enquanto as demais vão até 09/2026). Em vez de a loja sumir do painel, usamos o **último mês
> disponível dela**, marcado com `*` e explicado no rodapé.

**De % para kWh — fator exato.** Na Celesc, cada beneficiária recebe `excedentes distribuídos da geradora × participação`,
então `fator = excedentes distribuídos ÷ 100` (kWh por 1%) é **exato**, não empírico. O fator empírico do PR
(`Σ injetada ÷ Σ %`) fica só como reserva. A variação (kWh) é calculada contra `% Atual × fator`, e não contra a
injeção lida — assim um mês cujo crédito ainda não foi lançado (injeção 0) não distorce a recomendação.

**% Ideal** — pensada para o ajuste da participação no portal da Celesc **1× ou 2× por ano**
(`HORIZONTE_MESES` = 12 ou 6). Redistribui a alocação total (`Σ percentual`) com base no **consumo médio**
dos últimos `JANELA_MEDIA_MESES` meses de cada loja; o total é **conservado** e a geradora fica com 0.

- **Estratégia C** (`CONSIDERAR_SALDO=True`, padrão): `necessidade = max(0, (HORIZONTE_MESES + MESES_BUFFER) ×
  consumo_médio − saldo)`; `% ∝ necessidade`. Quem tem sobra recebe menos até consumi-la no horizonte.
- **Estratégia A** (`CONSIDERAR_SALDO=False`): `% = Σpct × consumo_médio ÷ Σ consumo_médio` — cobertura igual para todas.
- `PISO_PCT` garante uma % mínima por filial (o excedente é renormalizado).

Se a geração não cobre todo o consumo, **não é possível zerar todas as filiais** — o rateio só reparte a
cobertura existente. O rodapé do painel mostra a cobertura calculada.

**Geradora** recebe o chip neutro *Geradora* (saldo 0 nela é o esperado: distribui todo o excedente).

`MES_REFERENCIA_PAINEL = "2026-08"` fixa o painel num mês já fechado, se o mais recente estiver incompleto.

In [ ]:
FATOR_KWH_POR_PCT = None   # preenchido por agregar_workbook()


def _num(v) -> float:
    if v is None:
        return 0.0
    if isinstance(v, (int, float)):
        return float(v)
    return limpar_numero(v)


def _media_recente(meses: dict) -> float:
    """Média dos últimos JANELA_MEDIA_MESES meses (None = todo o histórico)."""
    refs = sorted(meses)
    if JANELA_MEDIA_MESES:
        refs = refs[-int(JANELA_MEDIA_MESES):]
    return statistics.mean(meses[r] for r in refs) if refs else 0.0


def agregar_workbook(xlsx_path: Path = XLSX_OUT):
    """Lê o Demonstrativo e devolve (dados, mes_referencia). Loja sem o mês mais recente usa o
    último mês dela e recebe `defasada=True`. O consumo médio usa os últimos
    `JANELA_MEDIA_MESES` meses (base da sugestão de % para o horizonte de revisão)."""
    if not Path(xlsx_path).exists():
        xlsx_path = XLSX_PATH
    wb = openpyxl.load_workbook(io.BytesIO(Path(xlsx_path).read_bytes()), data_only=False)
    ws = wb[ABA_MESTRE]
    linhas, maxmes = [], None
    for r in range(2, ws.max_row + 1):
        loja = ws.cell(r, COL_IDX["nome_loja"]).value
        ref = ws.cell(r, COL_IDX["data_referencia"]).value
        if not loja or not hasattr(ref, "year"):
            continue
        if MES_REFERENCIA_PAINEL and ref.strftime("%Y-%m") > MES_REFERENCIA_PAINEL:
            continue                                    # painel fixado num mês anterior
        linhas.append(dict(loja=str(loja).strip(), ref=ref,
                           dist=_num(ws.cell(r, COL_IDX["excedentes_distribuidos"]).value),
                           consumo=_num(ws.cell(r, COL_IDX["consumo"]).value),
                           cred=_num(ws.cell(r, COL_IDX["cred_receb_fp"]).value),
                           pct=_num(ws.cell(r, COL_IDX["percentual"]).value),
                           saldo=_num(ws.cell(r, COL_IDX["saldo_final_fp"]).value)))
        maxmes = ref if maxmes is None or ref > maxmes else maxmes
    if not linhas:
        raise RuntimeError("Nenhuma linha de dados no Demonstrativo.")

    cons_mes = defaultdict(lambda: defaultdict(float))
    cred_mes = defaultdict(lambda: defaultdict(float))
    ult_loja = {}
    for x in linhas:
        cons_mes[x["loja"]][x["ref"]] += x["consumo"]
        cred_mes[x["loja"]][x["ref"]] += x["cred"]
        ult_loja[x["loja"]] = max(ult_loja.get(x["loja"], x["ref"]), x["ref"])

    ult = {}
    for x in linhas:
        if x["ref"] == ult_loja[x["loja"]]:
            d = ult.setdefault(x["loja"], dict(consumo=0.0, cred=0.0, saldo=0.0, pct=0.0))
            d["consumo"] += x["consumo"]; d["cred"] += x["cred"]; d["saldo"] += x["saldo"]
            d["pct"] = max(d["pct"], x["pct"])

    dados = []
    for loja in sorted(ult):
        d = ult[loja]
        dados.append(dict(loja=loja, consumo=d["consumo"], injetada=d["cred"], saldo=d["saldo"],
                          pct_atual=d["pct"],
                          media_inj=statistics.mean(cred_mes[loja].values()),
                          consumo_medio=_media_recente(cons_mes[loja]),
                          ref_loja=ult_loja[loja], defasada=ult_loja[loja] < maxmes))
    for d in dados:
        if d["defasada"]:
            logger.warning("Painel | %s sem dados de %s — usando %s.", d["loja"],
                           maxmes.strftime("%m/%Y"), d["ref_loja"].strftime("%m/%Y"))
    logger.info("Etapa 3 | %d lojas agregadas (mês de referência %s).", len(dados), maxmes.strftime("%m/%Y"))
    # fator exato Celesc: kWh que cada 1% de participação recebeu = excedentes distribuídos da geradora / 100
    dist_mes = sum(x["dist"] for x in linhas if x["ref"] == maxmes and _eh_geradora(x["loja"]))
    global FATOR_KWH_POR_PCT
    FATOR_KWH_POR_PCT = dist_mes / 100 if dist_mes else None
    return dados, maxmes

In [ ]:
def classificar(saldo: float):
    """Devolve (rótulo, cor) do status do saldo conforme os limiares de CONFIG."""
    if saldo > LIM_SUPERAVIT:
        return "Superávit", "#1a7f5a"
    if saldo > LIM_CONFORTAVEL:
        return "Confortável", "#2f7fb0"
    if saldo >= LIM_BAIXO:
        return "Saldo baixo", "#c98a1a"
    return "Crítico / Zero", "#b23b3b"


COR_GERADORA = "#6b7b7b"


def _eh_geradora(nome: str) -> bool:
    # pelo rótulo (GERADORA) ou pela UC geradora no nome (nomes da aba Unidades Consumidoras)
    return (TAG_GERADORA.upper() in str(nome).upper()
            or any(str(uc) in str(nome) for uc in UCS_GERADORAS))


def _aplicar_piso(pesos: dict, total: float, piso: float) -> dict:
    """Reparte `total` proporcionalmente a `pesos`, garantindo `piso` a cada loja.
    Lojas que ficariam abaixo do piso são fixadas nele e o restante é renormalizado
    entre as demais (iterativo). Conserva a soma = total."""
    fixos, livres = {}, dict(pesos)
    while True:
        resto = total - piso * len(fixos)
        soma = sum(livres.values())
        if not livres or resto <= 0:
            base = total / max(len(pesos), 1)
            return {l: base for l in pesos}          # piso inviável: divide igualmente
        parc = {l: (resto * w / soma if soma else resto / len(livres)) for l, w in livres.items()}
        abaixo = [l for l, v in parc.items() if v < piso]
        if not piso or not abaixo:
            return {**{l: piso for l in fixos}, **parc}
        for l in abaixo:
            fixos[l] = piso
            livres.pop(l)


def _arredondar_conservando(valores: dict, casas: int = 2) -> dict:
    """Arredonda para `casas` decimais mantendo a soma (método do maior resto), para que
    a soma das % digitadas no portal seja exatamente o total atual."""
    q = 10 ** casas
    alvo = round(sum(valores.values()) * q)
    base = {k: int(v * q) for k, v in valores.items()}          # trunca
    falta = alvo - sum(base.values())
    for k in sorted(valores, key=lambda k: valores[k] * q - base[k], reverse=True)[:max(falta, 0)]:
        base[k] += 1
    return {k: v / q for k, v in base.items()}


def sugerir_pct_ideal(dados: list, rateio: dict | None = None) -> list:
    """Sugere a % de participação de cada filial para valer até a próxima revisão no portal
    da Celesc (feita 1-2x por ano), com base no CONSUMO MÉDIO de cada loja.

    O total de alocação (Σ percentual das beneficiárias) é CONSERVADO — a regulagem só
    reparte a geração existente; a geradora recebe % ideal = 0.

      - Estratégia A (`CONSIDERAR_SALDO=False`) — uniforme:
            pct_loja = Σpct × consumo_medio_loja / Σ consumo_medio
        Todas as lojas ficam com a mesma cobertura (injetada/consumo); ignora o saldo.
      - Estratégia C (`CONSIDERAR_SALDO=True`) — necessidade descontando o saldo:
            necessidade = max(0, (HORIZONTE_MESES + MESES_BUFFER) × consumo_medio − saldo)
            pct_loja ∝ necessidade  (renormalizado para conservar Σpct)
        Quem tem sobra acumulada recebe menos, até consumi-la no horizonte.
    `PISO_PCT` garante uma % mínima por filial (excedente renormalizado entre as demais).

    Conversão para ENERGIA (kWh). Celesc: fator exato = excedentes distribuídos da geradora
    ÷ 100. Sem esse dado, cai no fator empírico `Σ energia injetada / Σ percentual`.
      - `inj_ideal_kwh` = energia estimada que a % Ideal produziria (≈ pct_ideal × fator);
      - `delta_kwh`     = inj_ideal_kwh − pct_atual × fator (neutro a mês com crédito não lançado).
    Como o total é conservado, Σ delta_kwh ≈ 0.

    Registra em `RESUMO_RATEIO` a cobertura global (Σpct × fator / Σ consumo médio): se
    < 100%, NÃO é possível zerar todas as filiais ao mesmo tempo."""
    global RESUMO_RATEIO
    benef = [d for d in dados if not _eh_geradora(d["loja"])]
    total_pct = sum(d["pct_atual"] for d in benef)
    total_inj = sum(d["injetada"] for d in benef)
    total_cons_med = sum(d["consumo_medio"] for d in benef)
    if FATOR_KWH_POR_PCT:                                     # Celesc: exato (distribuído pela geradora / 100)
        fator = FATOR_KWH_POR_PCT
        logger.info("Fator kWh/%% = %.1f (excedentes distribuídos da geradora ÷ 100).", fator)
    else:                                                     # reserva: empírico, como no PR
        fator = (total_inj / total_pct) if total_pct else 0.0
        logger.info("Fator kWh/%% = %.1f (empírico: Σ injetada ÷ Σ percentual).", fator)

    if ESTRATEGIA_RATEIO == "W" and rateio:
        estrategia = "W"   # % calculado pelo rateio anual (Etapa 4)
        alocado = {d["loja"]: float(rateio["tabela"]["pct_ideal"].get(d["loja"], 0.0)) for d in benef}
    else:
        estrategia = "C" if CONSIDERAR_SALDO else "A"
        pesos = {d["loja"]: d["consumo_medio"] for d in benef}      # estratégia A
        if CONSIDERAR_SALDO:
            meses = HORIZONTE_MESES + MESES_BUFFER
            nec = {d["loja"]: max(0.0, meses * d["consumo_medio"] - d["saldo"]) for d in benef}
            if sum(nec.values()) > 0:
                pesos = nec
            else:
                logger.warning("Estratégia C: nenhuma loja com necessidade no horizonte — usando A.")
                estrategia = "A"
        alocado = _arredondar_conservando(_aplicar_piso(pesos, total_pct, PISO_PCT)) if benef else {}

    for d in dados:
        if _eh_geradora(d["loja"]):
            d["pct_ideal"] = 0.0
            d["inj_ideal_kwh"] = 0.0
            d["delta_kwh"] = 0.0
        else:
            d["pct_ideal"] = alocado[d["loja"]]
            d["inj_ideal_kwh"] = round(d["pct_ideal"] * fator)
            # delta contra o que a % ATUAL rende com o mesmo fator (neutro a mês com crédito não lançado)
            d["delta_kwh"] = round(d["inj_ideal_kwh"] - d["pct_atual"] * fator)

    cobertura = (total_pct * fator / total_cons_med) if total_cons_med else 0.0
    RESUMO_RATEIO = dict(estrategia=estrategia, horizonte=HORIZONTE_MESES, buffer=MESES_BUFFER,
                         piso=PISO_PCT, total_pct=total_pct, fator=fator, cobertura=cobertura,
                         soma_ideal=round(sum(d["pct_ideal"] for d in benef), 2))
    if estrategia == "W":
        RESUMO_RATEIO.update(k=rateio["k"], cobertura_anual=rateio["cobertura_estrutural"],
                             inicio=rateio["parametros"]["Início do ciclo"])
    logger.info("%% ideal | estratégia %s, horizonte %d meses (+%.1f), Σpct %.2f → Σideal %.2f, "
                "fator ≈ %.1f kWh/%%", estrategia, HORIZONTE_MESES,
                MESES_BUFFER if estrategia == "C" else 0, total_pct,
                RESUMO_RATEIO["soma_ideal"], fator)
    if cobertura < 1:
        logger.warning("%% ideal | a geração cobre só ~%.0f%% do consumo médio das filiais — "
                       "não é possível zerar todas; o rateio apenas reparte a cobertura existente.",
                       cobertura * 100)
    return dados


RESUMO_RATEIO: dict = {}

### 4.1 Renderização do painel (`matplotlib`)

In [ ]:
import datetime as _dt

# Paleta do painel
COR_FUNDO   = "#f4f6f8"
COR_HEADER  = "#0f3d3e"
COR_HEADER2 = "#155e63"
COR_TXT     = "#1c2b2b"
COR_LINHA_A = "#ffffff"
COR_LINHA_B = "#eef2f3"
COR_KPI     = "#ffffff"


def _fmt(n) -> str:
    """Formata número inteiro com ponto de milhar (padrão pt-BR)."""
    try:
        return f"{float(n):,.0f}".replace(",", ".")
    except Exception:
        return str(n)


def _fmt_sinal(n) -> str:
    """Número com sinal explícito: +57 / −13 / ±0."""
    n = round(float(n))
    return ("+" if n > 0 else ("\u2212" if n < 0 else "\u00b1")) + _fmt(abs(n))


def _wrap(s: str, width: int) -> str:
    """Quebra uma lista separada por vírgulas em várias linhas de largura ~width."""
    out, line = [], ""
    for w in s.split(", "):
        if len(line) + len(w) + 2 > width:
            out.append(line.rstrip(", ")); line = ""
        line += w + ", "
    out.append(line.rstrip(", "))
    return "\n".join([x for x in out if x])


def gerar_painel(dados: list, maxmes, saida_base: Path = PAINEL_OUT):
    """Desenha e salva o painel executivo. Devolve (caminho_png, caminho_pdf)."""
    dados = sorted(dados, key=lambda d: d["loja"])
    n = len(dados)
    row_h  = 0.30
    fig_h  = 6.0 + n * 0.34
    fig = plt.figure(figsize=(13.6, fig_h), dpi=140)
    fig.patch.set_facecolor(COR_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    H = fig_h
    def y(px): return 1 - px / H          # posição vertical em polegadas a partir do topo

    # ----- cabeçalho -----
    ax.add_patch(FancyBboxPatch((0.03, y(1.15)), 0.94, 1.05 / H,
                 boxstyle="round,pad=0.002,rounding_size=0.006",
                 fc=COR_HEADER, ec="none", transform=ax.transAxes, zorder=2))
    ax.text(0.06, y(0.52), "\u2600  Painel de Energia Injetada \u2014 Geração Distribuída Celesc",
            fontsize=20, fontweight="bold", color="white", va="center", zorder=3)
    ax.text(0.06, y(0.88), f"{EMPRESA}  \u00b7  {UF}",
            fontsize=11, color="#cfe3e3", va="center", zorder=3)
    ref_txt = f"{MESES_BR.get(maxmes.month, maxmes.month)}/{maxmes.year}" if hasattr(maxmes, "month") else str(maxmes)
    ax.text(0.945, y(0.50), ref_txt.upper(), fontsize=17, fontweight="bold",
            color="white", ha="right", va="center", zorder=3)
    ax.text(0.945, y(0.85), "Mês de referência", fontsize=9, color="#cfe3e3",
            ha="right", va="center", zorder=3)

    # ----- KPIs (excluem a geradora) -----
    benef = [d for d in dados if not _eh_geradora(d["loja"])]
    tot_cons = sum(d["consumo"] for d in benef)
    tot_inj  = sum(d["injetada"] for d in benef)
    cob = (tot_inj / tot_cons * 100) if tot_cons else 0
    n_sup = sum(1 for d in benef if d["status"] == "Superávit")
    n_cri = sum(1 for d in benef if d["status"] == "Crítico / Zero")
    kpis = [
        ("Consumo total",      f"{_fmt(tot_cons)} kWh", COR_HEADER2),
        ("Energia injetada",   f"{_fmt(tot_inj)} kWh",  "#1a7f5a"),
        ("Cobertura média",    f"{cob:.0f}%",           "#2f7fb0"),
        ("Lojas c/ superávit", str(n_sup),              "#1a7f5a"),
        ("Lojas críticas",     str(n_cri),              "#b23b3b"),
    ]
    kx0, kw, kgap = 0.03, 0.182, 0.0075
    ky_top, kh = 1.45, 1.0
    for i, (lab, val, cor) in enumerate(kpis):
        x = kx0 + i * (kw + kgap)
        ax.add_patch(FancyBboxPatch((x, y(ky_top + kh)), kw, kh / H,
                     boxstyle="round,pad=0.002,rounding_size=0.008",
                     fc=COR_KPI, ec="#d9e2e4", lw=1, transform=ax.transAxes, zorder=2))
        ax.add_patch(plt.Rectangle((x, y(ky_top + kh)), 0.006, kh / H, fc=cor, ec="none",
                     transform=ax.transAxes, zorder=3))
        ax.text(x + 0.02, y(ky_top + 0.30), val, fontsize=15.5, fontweight="bold",
                color=cor, va="center", zorder=3)
        ax.text(x + 0.02, y(ky_top + 0.72), lab, fontsize=9, color="#5a6b6b",
                va="center", zorder=3)

    # ----- cabeçalho da tabela -----
    cols = [
        ("Filial / Loja",        0.045, "left"),
        ("Consumida\n(kWh)",     0.40,  "right"),
        ("Injetada\n(kWh)",      0.475, "right"),
        ("Média Inj.\n(~kWh)",   0.55,  "right"),
        ("Saldo\n(kWh)",         0.625, "right"),
        ("% Atual",              0.685, "right"),
        ("% Ideal",              0.745, "right"),
        ("Inj. p/ Ideal\n(~kWh)",0.845, "right"),
        ("Status do Saldo",      0.868, "left"),
    ]
    yh = 2.7
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.5)), 0.94, 0.5 / H, fc=COR_HEADER2, ec="none",
                 transform=ax.transAxes, zorder=2))
    for name, cx, al in cols:
        ax.text(cx, y(yh + 0.25), name, fontsize=8.4, fontweight="bold", color="white",
                ha=al, va="center", zorder=3, linespacing=0.9)

    # ----- linhas -----
    y0 = yh + 0.5
    for i, d in enumerate(dados):
        yy = y0 + i * row_h
        band = COR_LINHA_A if i % 2 == 0 else COR_LINHA_B
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H, fc=band, ec="none",
                     transform=ax.transAxes, zorder=1))
        yc = y(yy + row_h / 2)
        ax.text(0.045, yc, d["loja"] + (f'  * ({d["ref_loja"].strftime("%m/%Y")})' if d.get("defasada") else ""), fontsize=8.6, color=COR_TXT, va="center", ha="left", zorder=3)
        ax.text(0.40,  yc, _fmt(d["consumo"]),  fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.475, yc, _fmt(d["injetada"]), fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.55,  yc, _fmt(d["media_inj"]),fontsize=8.6, va="center", ha="right", color="#5a6b6b", zorder=3)
        ax.text(0.625, yc, _fmt(d["saldo"]),    fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.685, yc, f'{d["pct_atual"]:.2f}', fontsize=8.6, va="center", ha="right", zorder=3)
        if _eh_geradora(d["loja"]):
            ax.text(0.745, yc, "\u2014", fontsize=8.6, va="center", ha="right", color="#9aa7a7", zorder=3)
            ax.text(0.828, yc, "\u2014", fontsize=8.6, va="center", ha="right", color="#9aa7a7", zorder=3)
        else:
            delta = d["pct_ideal"] - d["pct_atual"]
            seta  = "\u25b2" if delta > 0.05 else ("\u25bc" if delta < -0.05 else "=")
            cseta = "#1a7f5a" if delta > 0.05 else ("#b23b3b" if delta < -0.05 else "#5a6b6b")
            ax.text(0.745, yc, f'{d["pct_ideal"]:.2f}', fontsize=8.6, fontweight="bold",
                    va="center", ha="right", zorder=3)
            ax.text(0.751, yc, seta, fontsize=6.6, va="center", ha="left", color=cseta, zorder=3)
            # energia que a % Ideal produziria + variação (kWh) em relação à atual
            ax.text(0.828, yc, _fmt(d["inj_ideal_kwh"]), fontsize=8.6, fontweight="bold",
                    va="center", ha="right", zorder=3)
            dk = d["delta_kwh"]
            cdk = "#1a7f5a" if dk > 0 else ("#b23b3b" if dk < 0 else "#5a6b6b")
            sinal = "+" if dk > 0 else ("\u2212" if dk < 0 else "\u00b1")
            ax.text(0.862, yc, f'{sinal}{_fmt(abs(dk))}', fontsize=6.4, va="center", ha="right",
                    color=cdk, zorder=3)
        ax.add_patch(FancyBboxPatch((0.868, y(yy + row_h - 0.055)), 0.098, (row_h - 0.11) / H,
                     boxstyle="round,pad=0.001,rounding_size=0.010",
                     fc=d["cor"], ec="none", transform=ax.transAxes, zorder=3))
        ax.text(0.917, yc, d["status"], fontsize=7.1, fontweight="bold", color="white",
                va="center", ha="center", zorder=4)

    # ----- rodapé: recomendações -----
    fy = y0 + n * row_h + 0.35
    criticas = sorted([d for d in dados if d["status"] == "Crítico / Zero" and not _eh_geradora(d["loja"])],
                      key=lambda d: d["consumo"], reverse=True)
    superavit = sorted([d for d in dados if d["status"] == "Superávit" and not _eh_geradora(d["loja"])],
                       key=lambda d: d["saldo"], reverse=True)
    box_h = 2.15
    # caixa: atenção urgente
    ax.add_patch(FancyBboxPatch((0.03, y(fy + box_h)), 0.455, box_h / H,
                 boxstyle="round,pad=0.003,rounding_size=0.008",
                 fc="#fdecec", ec="#e6b3b3", lw=1, transform=ax.transAxes, zorder=2))
    ax.text(0.05, y(fy + 0.32), "\u26a0  Atenção urgente \u2014 reforçar alocação",
            fontsize=11, fontweight="bold", color="#b23b3b", va="center", zorder=3)
    nomes_c = ", ".join(f'{_rotulo_curto(d["loja"])} ({_fmt_sinal(d["delta_kwh"])})' for d in criticas[:10])
    if len(criticas) > 10:
        nomes_c += f"  (+{len(criticas) - 10} lojas)"
    ax.text(0.05, y(fy + 0.78), _wrap(nomes_c, 60) or "\u2014", fontsize=8.4, color="#7a2e2e",
            va="top", zorder=3, linespacing=1.35)
    ganho_tot = sum(d["delta_kwh"] for d in criticas)
    cap_c = (f"{len(criticas)} loja(s) com saldo zerado/negativo.\n"
             f"Variação total com a % Ideal: {_fmt_sinal(ganho_tot)} kWh/mês (valor por loja entre parênteses).")
    ax.text(0.05, y(fy + box_h - 0.18), cap_c,
            fontsize=7.4, color="#7a2e2e", style="italic", va="bottom", zorder=3,
            linespacing=1.3)
    # caixa: superávit
    ax.add_patch(FancyBboxPatch((0.515, y(fy + box_h)), 0.455, box_h / H,
                 boxstyle="round,pad=0.003,rounding_size=0.008",
                 fc="#e9f6ef", ec="#a9d9c1", lw=1, transform=ax.transAxes, zorder=2))
    ax.text(0.535, y(fy + 0.32), "\u2714  Superávit acumulado \u2014 reduzir alocação",
            fontsize=11, fontweight="bold", color="#1a7f5a", va="center", zorder=3)
    nomes_s = ", ".join(f'{_rotulo_curto(d["loja"])} ({_fmt_sinal(d["delta_kwh"])})' for d in superavit[:8])
    ax.text(0.535, y(fy + 0.78), _wrap(nomes_s, 58) or "\u2014", fontsize=8.4, color="#1c5c44",
            va="top", zorder=3, linespacing=1.35)
    cap_s = ("Créditos podem expirar (ReN ANEEL).\nReduzir a % destas lojas (% Ideal \u25bc) "
             "para consumir a sobra até a próxima revisão.")
    ax.text(0.535, y(fy + box_h - 0.18), cap_s,
            fontsize=7.4, color="#1c5c44", style="italic", va="bottom", zorder=3,
            linespacing=1.3)

    # ----- nota de rodapé -----
    hoje = _dt.date.today().strftime("%d/%m/%Y")
    rr = RESUMO_RATEIO or {}
    if rr:
        if rr["estrategia"] == "W":
            regra = (f"estratégia W \u2014 cobertura anual equalizada (k = {rr['k'] * 100:.1f}%, "
                     f"ciclo a partir de {rr['inicio']}; detalhes no relatório de rateio)")
        elif rr["estrategia"] == "C":
            regra = (f"estratégia C \u2014 necessidade de {rr['horizonte']} meses "
                     f"(+{rr['buffer']:.1f} de margem) de consumo médio menos o saldo")
        else:
            regra = "estratégia A \u2014 proporcional ao consumo médio (cobertura uniforme)"
        jan = f"{JANELA_MEDIA_MESES} meses" if JANELA_MEDIA_MESES else "todo o histórico"
        ax.text(0.03, y(fy + box_h + 0.62),
                f"% Ideal: {regra}; média de consumo: {jan}; Σ% conservada "
                f"({rr['total_pct']:.2f}); ~{rr['fator']:.0f} kWh por 1%.  "
                f"A geração cobre ~{rr['cobertura'] * 100:.0f}% do consumo médio"
                + (" \u2014 não é possível zerar todas as filiais." if rr["cobertura"] < 1 else "."),
                fontsize=7, color="#8a9797", va="center", zorder=3)
    if any(d.get("defasada") for d in dados):
        ax.text(0.03, y(fy + box_h + 0.89),
                "* Loja sem dados do mês de referência no relatório Celesc \u2014 exibido o último mês disponível (entre parênteses).",
                fontsize=7, color="#8a6d1a", va="center", zorder=3)
    ax.text(0.03, y(fy + box_h + 0.35),
            f"Gerado automaticamente em {hoje}  \u00b7  Fonte: {PDF_PATH.name}  \u00b7  "
            f"Regras: Superávit > {_fmt(LIM_SUPERAVIT)} kWh \u00b7 Confortável > {_fmt(LIM_CONFORTAVEL)} \u00b7 "
            f"Baixo \u2265 {LIM_BAIXO} \u00b7 Crítico/Zero < {LIM_BAIXO}  (parâmetros configuráveis)",
            fontsize=7, color="#8a9797", va="center", zorder=3)

    saida_base = Path(saida_base)
    png = saida_base.with_suffix(".png")
    pdf = saida_base.with_suffix(".pdf")
    fig.savefig(png, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    fig.savefig(pdf, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)
    logger.info("Etapa 3 | painel salvo em: %s  e  %s", png.name, pdf.name)
    return png, pdf


def gerar_relatorio_visual(xlsx_path: Path = XLSX_OUT, saida_base: Path = PAINEL_OUT,
                           rateio: dict | None = None):
    """Orquestra a Etapa 3: agrega -> % ideal -> classifica -> desenha painel."""
    dados, maxmes = agregar_workbook(xlsx_path)
    dados = sugerir_pct_ideal(dados, rateio)
    preso = rateio["tabela"]["saldo_preso"] if rateio else {}
    for d in dados:
        d["preso"] = float(preso.get(d["loja"], 0.0)) if rateio else 0.0
        # status pelo saldo UTILIZÁVEL: saldo preso em cadastro antigo não compensa a fatura
        d["status"], d["cor"] = (("Geradora", COR_GERADORA) if _eh_geradora(d["loja"])
                                 else classificar(d["saldo"] - d["preso"]))
        if d["preso"] > 0 and not _eh_geradora(d["loja"]) and d["saldo"] - d["preso"] <= LIM_CONFORTAVEL:
            d["status"], d["cor"] = "Saldo preso", "#6b5b95"
    png, pdf = gerar_painel(dados, maxmes, saida_base)
    return dados, maxmes, png, pdf

## Etapa 4 — Rateio anual (% fixo por 12 meses)

O % de cada loja só pode ser alterado **uma vez por ano**, então o objetivo não é o % "ótimo hoje", e sim
um % que **aguente os 12 meses** sem deixar saldo parado numa loja enquanto outra paga a fatura.
Este bloco é **o mesmo nos notebooks Copel, CELESC e Nelson**; só o *adaptador* (4.1) muda, porque cada
planilha guarda o `Demonstrativo` num formato diferente.

### Modelo

Com o % fixo `pᵢ` e a energia distribuída no mês `Gₜ`, o saldo de cada loja evolui assim:

```
Sᵢ,ₜ = Sᵢ,ₜ₋₁ + pᵢ·Gₜ − Compᵢ,ₜ        Compᵢ,ₜ = min(Sᵢ,ₜ₋₁ + pᵢ·Gₜ ; Uᵢ,ₜ)
Uᵢ,ₜ = max(0 ; Consumoᵢ,ₜ − CustoDispᵢ − GeraçãoPrópriaᵢ,ₜ)   ← consumo COMPENSÁVEL
Faturadoᵢ,ₜ = Uᵢ,ₜ − Compᵢ,ₜ
```

**Regra de rateio — cobertura equalizada ("water-filling"):**

```
aᵢ = max(0 ; k·Uᵢ,ano + Alvoᵢ − Sᵢ,início)      com k tal que  Σ aᵢ = G_ano
pᵢ = aᵢ / Σ aᵢ   (× Σ% atual, arredondado conservando a soma)
```

`k` é a **cobertura comum**: a maior cobertura que dá para garantir a todas as lojas ao mesmo tempo.
Lojas cujo saldo já passa de `k·U` recebem 0% e consomem o saldo; as demais ficam com a mesma cobertura.

### Tratamentos

| Tema | Como o bloco trata |
|---|---|
| Custo de disponibilidade | informado pelo adaptador (PDF), inferido dos dados (consumo − crédito usado) ou `CUSTO_DISP_PADRAO`; `CUSTO_DISP_MANUAL` sobrepõe tudo |
| Energia distribuível `G` | informada pelo adaptador (ex.: excedentes distribuídos) ou Σ créditos recebidos reescalada para 100% × `FATOR_GERACAO` |
| Mês incompleto | se as beneficiárias ainda não receberam o crédito do último mês, a análise para no último mês completo |
| Previsão de consumo | perfil dos últimos 12 meses × tendência amortecida (`TENDENCIA_*`) |
| Geração própria | UC que também é geradora só precisa do consumo que a própria geração não cobre |
| Saldo preso | linha de cadastro sem consumo com saldo, ou UC cujo saldo não é usado há `MESES_SALDO_PRESO` meses → não conta como saldo |
| Sazonalidade | a simulação começa em `MES_INICIO_CICLO`; o saldo-alvo inclui o **colchão sazonal** exigido por esse mês |
| Meses até a troca | simulados com o **% atual** para estimar o saldo no início do ciclo |
| Sobra estrutural | se geração + saldo passam do consumo, o relatório indica a parcela que poderia ser cedida a outras UCs |

### Validação

- **Comparação de métodos** (atual, legado 12,5×, proporcional, necessidade − saldo, cobertura equalizada,
  otimização numérica) com o % **fixo** sob os `CENARIOS` de estresse.
- **Backtest com origem móvel**: roda automaticamente quando houver **≥ 24 meses** de histórico.

### 4.0 Parâmetros do rateio anual

In [ ]:
# ---- Rateio anual (Etapas 4 e 5) — mesmos parâmetros nos notebooks Copel, CELESC e Nelson
ESTRATEGIA_RATEIO  = "W"      # "W" = cobertura anual equalizada (water-filling) — RECOMENDADO
                              # "C" / "A" = regras legadas da v1 (a % Ideal do painel volta a usá-las)
MES_INICIO_CICLO   = 10       # mês (1-12) em que o novo % passa a valer no site da distribuidora
FATOR_GERACAO      = 1.00     # geração do próximo ciclo ÷ últimos 12 meses (ex.: 0.995 = degradação dos módulos)
TENDENCIA_AMORTECIMENTO = 0.5 # 0 = ignora tendência de consumo; 1 = aplica a tendência inteira
TENDENCIA_LIMITES  = (0.85, 1.25)   # limites do fator de tendência por loja
USAR_BUFFER_SAZONAL = True    # saldo-alvo inclui o colchão sazonal exigido pelo mês de início do ciclo
MARGEM_SEGURANCA   = 0.03     # + fração do consumo anual compensável no saldo-alvo (erro de previsão)
CUSTO_DISP_PADRAO  = 100      # kWh/mês não compensável quando não há outra fonte (TR=100, BI=50, MO=30)
CUSTO_DISP_MANUAL: dict = {}  # {uc: kWh} — sobrepõe PDF / inferência
MESES_SALDO_PRESO  = 3        # nº de meses sem uso do saldo (com consumo) para classificá-lo como "preso"
PESO_SALDO_PRESO   = 0.5      # fração do saldo preso contada como saldo no rateio (só lojas ativas):
                              #   0 = o saldo nunca será liberado  -> a loja recebe % para cobrir tudo sozinha
                              #   1 = a distribuidora vai realocar o saldo antes do ciclo -> a loja recebe menos %
                              #   0.5 = meio-termo: limita o erro nos dois casos (falta vs. sobra de crédito)
AUTO_EXCLUIR_SEM_COMPENSACAO = True  # loja cujo saldo não é usado há MESES_SALDO_PRESO meses recebe 0%
EXCLUIR_DO_RATEIO: list = []  # nomes de loja forçados a 0% (ex.: loja que vai fechar)
RODAR_OTIMIZACAO   = True     # inclui o método "Otimização numérica" na comparação (mais lento)
CENARIOS = {                  # (fator geração, fator consumo) — o % é mantido FIXO em todos
    "Base":                        (1.00, 1.00),
    "Geração −15%":                (0.85, 1.00),
    "Geração +10%":                (1.10, 1.00),
    "Consumo +10%":                (1.00, 1.10),
    "Geração −15% e Consumo +10%": (0.85, 1.10),
}
ABAS_RATEIO = ["Rateio Anual", "Rateio Simulação", "Rateio Métodos", "Rateio Premissas"]
GERAR_ABAS_RATEIO = False     # False = o .xlsx não recebe as abas "Rateio *" (o rateio fica só no relatório PDF)

### 4.1 Adaptador Celesc SC — `entrada_rateio_celesc()`

Lê a aba-mestre `Demonstrativo` da planilha SC. A energia distribuível de cada mês é a coluna
**Excedentes distribuídos** da geradora (exata). Se no último mês a geradora já distribuiu mas o crédito
ainda não aparece nas beneficiárias, o bloco de rateio usa só até o último mês completo. O relatório Celesc
não informa o custo de disponibilidade: usa `CUSTO_DISP_PADRAO` (ajuste por UC em `CUSTO_DISP_MANUAL`).

In [ ]:
def entrada_rateio_celesc(xlsx_path: Path = XLSX_OUT) -> dict:
    """Converte a planilha SC para o formato padrão do bloco de rateio."""
    xlsx_path = Path(xlsx_path) if Path(xlsx_path).exists() else XLSX_PATH
    df = pd.read_excel(xlsx_path, sheet_name=ABA_MESTRE)
    df = df.dropna(subset=["nome_loja", "uc_conces", "data_referencia"]).copy()
    for c in COLS_NUMERICAS:
        df[c] = df[c].map(_num)
    ger = (df["tipo"].astype(str).str.lower() == "geradora") | df["nome_loja"].map(_eh_geradora)
    ref = pd.to_datetime(df["data_referencia"]).dt.to_period("M")
    std = pd.DataFrame({
        "loja": df["nome_loja"].astype(str).str.strip(), "uc": df["uc_conces"].astype("int64"), "ref": ref,
        "consumo": df["consumo"], "saldo_ant": df["saldo_anterior_fp"], "cred": df["cred_receb_fp"],
        "pct": df["percentual"], "saldo_fin": df["saldo_final_fp"], "geradora": ger.values,
        # a Celesc não fecha o saldo como a Copel: crédito usado = consumo − consumo faturado
        "usado": (df["consumo"] - df["consumo_faturado"]).clip(lower=0),
    })
    G = df.loc[ger, "excedentes_distribuidos"].groupby(ref[ger]).sum()
    return dict(df=std, G=G, sistema="Celesc · SC", fonte=f"{PDF_PATH.name} + aba {ABA_MESTRE}",
                portal="portal da Celesc")

### 4.2 Base de dados do rateio (formato padrão, vindo do adaptador)

O adaptador entrega um `DataFrame` com **uma linha por UC × seção × mês** e as colunas
`loja, uc, ref, consumo, saldo_ant, cred, pct, saldo_fin, geradora` (+ opcionais: `G` = energia
distribuível por mês, `cd` = custo de disponibilidade por UC, `geracao_propria`, `novas`, `alertas`).

In [ ]:
import io as _io
import numpy as np
import datetime as dt
from pathlib import Path
try:
    from scipy.optimize import minimize
except Exception:                                   # noqa: BLE001
    minimize = None

MESES_ABREV = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]


def _rotulo_mes(m: int) -> str:
    return MESES_ABREV[(m - 1) % 12]


def _r_fmt(n) -> str:
    try:
        return f"{float(n):,.0f}".replace(",", ".")
    except (TypeError, ValueError):
        return str(n)


def _r_fmt_sinal(n) -> str:
    n = round(float(n))
    return ("+" if n > 0 else ("−" if n < 0 else "±")) + _r_fmt(abs(n))


def _r_arred(valores: dict, casas: int = 2) -> dict:
    """Arredonda mantendo a soma (maior resto) — a soma digitada no site fecha exata."""
    q = 10 ** casas
    alvo = round(sum(valores.values()) * q)
    base = {k: int(v * q) for k, v in valores.items()}
    falta = alvo - sum(base.values())
    for k in sorted(valores, key=lambda k: valores[k] * q - base[k], reverse=True)[:max(falta, 0)]:
        base[k] += 1
    return {k: v / q for k, v in base.items()}


def _inferir_cd(por_uc: pd.DataFrame) -> dict:
    """Custo de disponibilidade pelos dados: mediana de (consumo − crédito usado) nos meses em que sobrou
    saldo (o consumo foi compensado até o limite), aproximada para 0/30/50/100 kWh."""
    ok = por_uc[(por_uc["saldo"] > 0) & (por_uc["consumo"] > 0) & (por_uc["usado"] > 0)]
    res = {}
    for uc, g in ok.groupby("uc"):
        v = float((g["consumo"] - g["usado"]).clip(lower=0).median())
        res[uc] = float(min([0, 30, 50, 100], key=lambda x: abs(x - v)))
    return res


def montar_base_rateio(entrada: dict) -> dict:
    """Séries mensais por loja a partir do formato padrão do adaptador."""
    df = entrada["df"].copy()
    df["ref"] = pd.PeriodIndex(df["ref"], freq="M")
    df["uc"] = df["uc"].astype("int64")
    for c in ["consumo", "saldo_ant", "cred", "pct", "saldo_fin"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    ger = df["geradora"].astype(bool)
    b = df[~ger].copy()
    meses_all = pd.period_range(b["ref"].min(), b["ref"].max(), freq="M")
    alertas = list(entrada.get("alertas") or [])

    # ---- energia distribuível e último mês COMPLETO (crédito já lançado nas beneficiárias)
    G_dist = b.groupby("ref")["cred"].sum().reindex(meses_all).fillna(0.0)
    G_ext = entrada.get("G")
    ult = meses_all[-1]
    if G_ext is not None:
        G_ext = pd.Series(G_ext).copy()
        G_ext.index = pd.PeriodIndex(G_ext.index, freq="M")
        ult_g = G_ext.index.max()                               # último mês com dado da geradora
        G_ext = G_ext.groupby(level=0).sum().reindex(meses_all).fillna(0.0)
        # mês além do último da geradora (só parte das beneficiárias já tem o mês) não conta como completo
        completos = [m for m in meses_all
                     if m <= ult_g and (G_ext[m] <= 0 or G_dist[m] >= 0.5 * G_ext[m])]
        ult = max(completos) if completos else ult
        if ult < meses_all[-1]:
            alertas.append(dict(tipo="Mês incompleto", loja="—", kwh=float(G_ext[meses_all[-1]]),
                                detalhe=f"de {ult + 1} a {meses_all[-1]} o crédito da geradora ainda não aparece "
                                        f"em todas as beneficiárias — a análise usa até {ult}"))
    meses = pd.period_range(meses_all[0], ult, freq="M")
    b = b[b["ref"] <= ult]
    df = df[df["ref"] <= ult]

    if "usado" in b.columns and b["usado"].notna().any():          # informado pelo adaptador (ex.: Celesc)
        b["usado"] = pd.to_numeric(b["usado"], errors="coerce").fillna(0.0).clip(lower=0)
    else:                                                          # balanço: anterior + recebido − final
        b["usado"] = (b["saldo_ant"] + b["cred"] - b["saldo_fin"]).clip(lower=0)
    por_uc = (b.groupby(["loja", "uc", "ref"], as_index=False)
                .agg(consumo=("consumo", "sum"), usado=("usado", "sum"),
                     saldo=("saldo_fin", "sum"), cred=("cred", "sum")))

    # ---- custo de disponibilidade: manual > adaptador > inferido > padrão
    cd_ent = dict(entrada.get("cd") or {})
    cd_inf = _inferir_cd(por_uc) if entrada.get("inferir_cd") else {}
    origem_cd = {}
    def _cd(uc):
        for fonte, d in (("manual", CUSTO_DISP_MANUAL), ("PDF", cd_ent), ("inferido", cd_inf)):
            if uc in d and d[uc] is not None and not pd.isna(d[uc]):
                origem_cd[uc] = fonte
                return float(d[uc])
        origem_cd[uc] = "padrão"
        return float(CUSTO_DISP_PADRAO)
    por_uc["cd"] = por_uc["uc"].map(_cd)

    # ---- geração própria (UC que também é geradora em outro demonstrativo)
    proprio = entrada.get("geracao_propria") or {}
    por_uc["proprio"] = 0.0
    for uc, serie in proprio.items():
        s = pd.Series(serie).copy(); s.index = pd.PeriodIndex(s.index, freq="M")
        m = por_uc["uc"] == uc
        por_uc.loc[m, "proprio"] = por_uc.loc[m, "ref"].map(s.groupby(level=0).sum()).fillna(0.0).values
    por_uc["U"] = (por_uc["consumo"] - por_uc["cd"] - por_uc["proprio"]).clip(lower=0)

    def _pivot(frame, col):
        return (frame.pivot_table(index="loja", columns="ref", values=col, aggfunc="sum")
                     .reindex(columns=meses).fillna(0.0))
    C, U, S_mes = _pivot(por_uc, "consumo"), _pivot(por_uc, "U"), _pivot(por_uc, "saldo")

    # ---- energia distribuível a 100% das beneficiárias
    pct_mes = b.groupby(["ref", "uc"])["pct"].max().groupby("ref").sum().reindex(meses)
    if G_ext is not None:
        G = G_ext.reindex(meses).fillna(0.0)
    else:
        G = (G_dist.reindex(meses) * 100 / pct_mes.where(pct_mes > 0)).fillna(G_dist.reindex(meses))

    # ---- saldo preso (a) linha de cadastro sem consumo e com saldo no último mês
    lb = b[b["ref"] == ult]
    preso_a = (lb[(lb["consumo"] <= 0) & (lb["saldo_fin"] > 0) & (lb["cred"] <= 0)]
               .groupby("loja")["saldo_fin"].sum())
    # ---- (b) UC com consumo compensável e saldo, mas sem uso nos últimos N meses
    jan = por_uc[por_uc["ref"].isin(meses[-MESES_SALDO_PRESO:])]
    agg = jan.groupby(["loja", "uc"]).agg(U=("U", "sum"), usado=("usado", "sum"))
    s_ult = por_uc[por_uc["ref"] == ult].set_index(["loja", "uc"])["saldo"].reindex(agg.index).fillna(0.0)
    cond = (agg["U"] > 0) & (agg["usado"] <= 0.01 * agg["U"]) & (s_ult > 0)
    preso_b = s_ult[cond].groupby(level=0).sum()
    sem_comp = sorted(preso_b.index)

    lojas = list(C.index)
    S_total = S_mes[ult].reindex(lojas).fillna(0.0)
    preso = pd.concat([preso_a, preso_b], axis=1).max(axis=1).reindex(lojas).fillna(0.0)
    preso = np.minimum(preso, S_total)
    S_util = (S_total - preso).clip(lower=0)
    pct_atual = (lb.groupby(["loja", "uc"])["pct"].max()          # 1 linha "real" por UC
                   .groupby(level=0).sum().reindex(lojas).fillna(0.0))  # loja com 2+ UCs soma
    ucs = b.groupby("loja")["uc"].unique().map(lambda a: ", ".join(map(str, sorted(a))))

    # ---- alertas
    for l in lojas:
        if preso[l] > 0:
            motivo = (f"saldo não é usado há ≥ {MESES_SALDO_PRESO} meses apesar do consumo"
                      if l in sem_comp else "saldo em cadastro antigo/inativo (sem consumo)")
            alertas.append(dict(tipo="Saldo preso", loja=l, kwh=float(preso[l]),
                                detalhe=motivo + " — solicitar realocação/regularização à distribuidora"))
    for l in sem_comp:
        alertas.append(dict(tipo="Sem compensação", loja=l, kwh=float(C.loc[l].iloc[-MESES_SALDO_PRESO:].sum()),
                            detalhe="consumo não está sendo compensado; % ideal = 0 até regularizar"
                                    if AUTO_EXCLUIR_SEM_COMPENSACAO else "consumo não está sendo compensado"))
    ult12 = C.iloc[:, -12:]
    for l in lojas:
        n_obs = int((ult12.loc[l] > 0).sum())
        if 0 < n_obs < min(12, ult12.shape[1]):
            alertas.append(dict(tipo="Histórico incompleto", loja=l, kwh=None,
                                detalhe=f"só {n_obs} de 12 meses com consumo — meses faltantes estimados pela média"))
    pct_loja_mes = b.groupby(["loja", "ref"])["pct"].max().unstack().reindex(index=lojas, columns=meses).fillna(0)
    cred_loja_mes = b.groupby(["loja", "ref"])["cred"].sum().unstack().reindex(index=lojas, columns=meses).fillna(0)
    for l in lojas:
        falhas = [m for m in meses[-12:] if pct_loja_mes.loc[l, m] > 0 and G[m] > 0 and cred_loja_mes.loc[l, m] <= 0]
        if len(falhas) >= 3:
            alertas.append(dict(tipo="Crédito não recebido", loja=l, kwh=None,
                                detalhe=f"com % > 0, não recebeu crédito em {len(falhas)} dos últimos 12 meses "
                                        f"({', '.join(str(m) for m in falhas[:6])}{'…' if len(falhas) > 6 else ''}) — conferir com a distribuidora"))
    lojas_por_uc = por_uc.groupby("uc")["loja"].first()
    for uc in proprio:
        if uc in lojas_por_uc.index:
            l = lojas_por_uc[uc]
            cob = float(por_uc.loc[por_uc["uc"] == uc, "proprio"].sum())
            alertas.append(dict(tipo="Geração própria", loja=l, kwh=cob,
                                detalhe="a UC também é geradora; só recebe créditos para o consumo que a própria "
                                        "geração não cobre (kWh = geração própria nos meses analisados)"))
    for n in entrada.get("novas") or []:
        alertas.append(dict(tipo="UC nova sem histórico", loja=f"UC {n['uc']}", kwh=None,
                            detalhe=f"{n.get('endereco') or ''} — fora do rateio; definir % por loja análoga"))
    g_df = df[ger]
    saldo_ger = float(g_df[g_df["ref"] == g_df["ref"].max()]["saldo_fin"].sum()) if not g_df.empty else 0.0
    if saldo_ger > 0:
        alertas.append(dict(tipo="Saldo na geradora", loja="GERADORA", kwh=saldo_ger,
                            detalhe="saldo acumulado na própria geradora não é distribuído às beneficiárias"))

    logger.info("Etapa 4 | base: %d loja(s) × %d meses (%s a %s); saldo utilizável %s kWh; preso %s kWh.",
                len(lojas), len(meses), meses[0], ult, _r_fmt(S_util.sum()), _r_fmt(preso.sum()))
    cd_mensal = por_uc.groupby("loja")["cd"].sum() / por_uc.groupby("loja")["ref"].nunique()
    return dict(meses=meses, ultimo=ult, C=C, U=U, S_mes=S_mes, G=G, G_dist=G_dist, pct_mes=pct_mes,
                S_total=S_total, preso=preso, S_util=S_util, pct_atual=pct_atual, ucs=ucs,
                sem_compensacao=sem_comp, alertas=alertas, saldo_geradora=saldo_ger, cd_mensal=cd_mensal,
                origem_cd=origem_cd)

### 4.3 Previsão, simulação mensal, cobertura equalizada e métodos comparados

In [ ]:
def projetar_consumo(U12: np.ndarray):
    """Previsão híbrida: perfil dos últimos 12 meses × fator de tendência amortecido.
    Meses iniciais sem consumo (loja/cadastro recente) são preenchidos com a média observada.
    g = média dos 6 meses mais recentes ÷ média dos 6 anteriores (proxy de crescimento — hipótese)."""
    arr = np.array(U12, dtype=float)
    for i in range(arr.shape[0]):
        obs = arr[i] > 0
        if 0 < obs.sum() < arr.shape[1]:
            primeiro = int(np.argmax(obs))
            if primeiro > 0:
                arr[i, :primeiro] = arr[i, obs].mean()
    g = arr[:, 6:].mean(1) / np.maximum(arr[:, :6].mean(1), 1.0)
    g = np.where(arr[:, :6].mean(1) > 0, g, 1.0)
    g = np.clip(1 + TENDENCIA_AMORTECIMENTO * (g - 1), *TENDENCIA_LIMITES)
    return arr * g[:, None], g


def simular(p, U, G, S0):
    """Aplica o % fixo mês a mês. Devolve (saldo[lojas×T], compensado, faturado, recebido)."""
    n, T = U.shape
    S = np.zeros((n, T)); comp = np.zeros((n, T)); rec = np.zeros((n, T))
    s = np.array(S0, dtype=float).copy()
    for t in range(T):
        rec[:, t] = p * G[t]
        disp = s + rec[:, t]
        comp[:, t] = np.minimum(disp, U[:, t])
        s = disp - comp[:, t]
        S[:, t] = s
    return S, comp, U - comp, rec


def water_filling(Uano, S0, Gano, alvo=0.0, ativos=None):
    """Maximiza a cobertura mínima: aᵢ = max(0, k·Uᵢ + alvoᵢ − Sᵢ), Σ aᵢ = Gano (busca binária em k)."""
    Uano = np.asarray(Uano, float); S0 = np.asarray(S0, float)
    alvo = np.broadcast_to(np.asarray(alvo, float), Uano.shape)
    ativos = np.ones_like(Uano, bool) if ativos is None else np.asarray(ativos, bool)
    if not ativos.any() or Uano[ativos].sum() <= 0:
        a = np.where(ativos, 1.0, 0.0)
        return (a / a.sum() if a.sum() else a), float("inf")
    lo, hi = 0.0, max(20.0, 2 * (Gano + S0[ativos].sum()) / Uano[ativos].sum() + 1)
    for _ in range(200):
        k = (lo + hi) / 2
        a = np.where(ativos, np.maximum(0, k * Uano + alvo - S0), 0.0)
        lo, hi = (k, hi) if a.sum() < Gano else (lo, k)
    return (a / a.sum() if a.sum() else a), k


def buffer_sazonal(G12, U12tot):
    """Colchão (em meses de consumo) que cada mês de início exige para não faltar crédito com % fixo,
    e o pico de saldo acumulado durante o ciclo. Linha i = i-ésimo mês da janela de 12 meses."""
    G12, U12tot = np.asarray(G12, float), np.asarray(U12tot, float)
    if G12.sum() <= 0 or U12tot.sum() <= 0:
        return np.zeros((12, 2))
    w = G12 / G12.sum(); u = U12tot / U12tot.sum()
    linhas = []
    for i in range(12):
        d = np.cumsum(np.roll(w - u, -i)) * 12
        linhas.append((max(0.0, -d.min()), max(0.0, d.max())))
    return np.array(linhas)


def metricas(p, U, G, S0, ativos):
    """Métricas objetivas de um % fixo aplicado por 12 meses (só lojas ativas)."""
    S, comp, fat, _ = simular(p, U, G, S0)
    if not ativos.any():
        ativos = np.ones_like(ativos, bool)
    U, S, comp, fat = U[ativos], S[ativos], comp[ativos], fat[ativos]
    Uano = np.maximum(U.sum(1), 1)
    cob = np.minimum(comp.sum(1) / Uano, 1.0)
    Sf = S[:, -1]
    minimo = max(0.0, U.sum() - np.sum(G) * p[ativos].sum() - np.asarray(S0)[ativos].sum())
    return {
        "Faturado (kWh)": fat.sum(),
        "Faturado acima do mínimo (kWh)": max(0.0, fat.sum() - minimo),
        "Saldo final total (kWh)": Sf.sum(),
        "Saldo ocioso >1 mês (kWh)": np.maximum(0, Sf - U[:, -1]).sum(),
        "Saldo final máx. (meses)": (Sf / (Uano / 12)).max(),
        "Lojas cobertura <90%": int((cob < 0.90).sum()),
        "Cobertura mín.": cob.min(),
        "Dispersão cobertura (dp)": cob.std(),
        "Pico de saldo total (kWh)": S.sum(0).max(),
    }


def otimizar(U, G, S0, p0, ativos, w_disp=20000.0):
    """Método 6: minimiza faturado + saldo final acima de 1 mês + w·var(cobertura) na simulação mensal."""
    if minimize is None or ativos.sum() < 2:
        return None
    idx = np.where(ativos)[0]
    Ua, S0a, Uano = U[idx], np.asarray(S0)[idx], np.maximum(U[idx].sum(1), 1)

    def custo(z):
        q = np.exp(z - z.max()); q /= q.sum()
        S, comp, fat, _ = simular(q, Ua, G, S0a)
        cob = comp.sum(1) / Uano
        return fat.sum() + np.maximum(0, S[:, -1] - Ua[:, -1]).sum() + w_disp * cob.var() * len(q)

    r = minimize(custo, np.log(np.maximum(p0[idx], 1e-6)), method="Powell",
                 options={"maxiter": 20000, "xtol": 1e-4})
    q = np.exp(r.x - r.x.max()); q /= q.sum()
    p = np.zeros_like(p0); p[idx] = q
    return p


def _normalizar(v, ativos):
    v = np.where(ativos, np.maximum(np.asarray(v, float), 0), 0.0)
    return v / v.sum() if v.sum() else v


def calcular_metodos(U, C12, Gano, S0, S_total, pct_atual, ativos, alvo):
    """Percentuais (frações, soma 1) de cada método comparado."""
    Uano = U.sum(1)
    Cm = C12.mean(1)
    P = {"Atual": _normalizar(pct_atual, np.ones_like(ativos))}
    P["Legado 12,5×"] = _normalizar(np.maximum(0, (HORIZONTE_MESES + MESES_BUFFER) * Cm - S_total), ativos)
    P["Proporcional ao consumo"] = _normalizar(Cm, ativos)
    P["Necessidade − saldo"] = _normalizar(np.maximum(0, Uano - S0), ativos)
    P["Cobertura equalizada (W)"], k = water_filling(Uano, S0, Gano, alvo, ativos)
    return P, k

### 4.4 Orquestrador da Etapa 4 — `calcular_rateio_anual()` (inclui backtest)

In [ ]:
def _ciclo(meses_12: pd.PeriodIndex, mes_inicio: int) -> list:
    """Índices (0-11) das colunas dos últimos 12 meses reordenados para começar em `mes_inicio`."""
    pos = {m.month: i for i, m in enumerate(meses_12)}
    return [pos[(mes_inicio - 1 + i) % 12 + 1] for i in range(12)]


def backtest_origem_movel(base: dict, ativos_nomes: list) -> pd.DataFrame | None:
    """Backtest real: para cada origem com 12 meses antes e 12 depois, calcula o % só com o passado,
    mantém fixo e aplica na geração/consumo REAIS. Requer ≥ 24 meses de histórico."""
    meses = base["meses"]
    if len(meses) < 24:
        logger.warning("Etapa 4 | backtest com origem móvel requer ≥ 24 meses (há %d). "
                       "Usando apenas simulação prospectiva + estresse.", len(meses))
        return None
    lojas = list(base["U"].index)
    ativos = np.array([l in ativos_nomes for l in lojas])
    Uall, Call, Sall = base["U"].to_numpy(float), base["C"].to_numpy(float), base["S_mes"].to_numpy(float)
    Gall = base["G"].to_numpy(float)
    pct = base["pct_atual"].to_numpy(float)
    linhas = []
    for o in range(12, len(meses) - 11):
        Up, _ = projetar_consumo(Uall[:, o - 12:o])
        S0 = Sall[:, o - 1]
        P, _ = calcular_metodos(Up, Call[:, o - 12:o], Gall[o - 12:o].sum(), S0, S0, pct, ativos, 0.0)
        for nome, p in P.items():
            linhas.append(dict(origem=str(meses[o]), metodo=nome,
                               **metricas(p, Uall[:, o:o + 12], Gall[o:o + 12], S0, ativos)))
    bt = pd.DataFrame(linhas)
    logger.info("Etapa 4 | backtest: %d origens × %d métodos.", bt["origem"].nunique(), bt["metodo"].nunique())
    return bt


def calcular_rateio_anual(entrada: dict) -> dict:
    """Etapa 4 completa. `entrada` vem do adaptador do notebook (formato padrão)."""
    base = montar_base_rateio(entrada)
    lojas = list(base["U"].index)
    n = len(lojas)
    meses12 = base["meses"][-12:]
    if len(meses12) < 12:
        raise RuntimeError(f"O rateio anual precisa de 12 meses de histórico (há {len(meses12)}).")

    excl = set(EXCLUIR_DO_RATEIO) | (set(base["sem_compensacao"]) if AUTO_EXCLUIR_SEM_COMPENSACAO else set())
    ativos = np.array([l not in excl for l in lojas])

    # ---- previsão (ordem de calendário dos últimos 12 meses) e alinhamento ao ciclo
    U12 = base["U"][meses12].to_numpy(float)
    C12 = base["C"][meses12].to_numpy(float)
    Uproj_cal, g = projetar_consumo(U12)
    G_cal = base["G"][meses12].to_numpy(float) * FATOR_GERACAO
    med = np.median(G_cal[G_cal > 0]) if (G_cal > 0).any() else 0.0
    for i in np.where(G_cal > 2.2 * med)[0] if med else []:
        viz = (G_cal[(i - 1) % 12] + G_cal[(i + 1) % 12]) / 2
        base["alertas"].append(dict(tipo="Geração atípica", loja="—", kwh=float(G_cal[i]),
                                    detalhe=f"{meses12[i]} tem {G_cal[i] / med:.1f}× a mediana mensal — na projeção "
                                            f"foi substituído pela média dos meses vizinhos ({_r_fmt(viz)} kWh)"))
        G_cal[i] = viz
    ordem = _ciclo(meses12, MES_INICIO_CICLO)
    Uc, Gc = Uproj_cal[:, ordem], G_cal[ordem]
    Gano = Gc.sum()

    pct_atual = base["pct_atual"].to_numpy(float)
    total_pct = pct_atual.sum()
    p_atual = pct_atual / total_pct if total_pct else np.full(n, 1 / n)
    G_benef = Gc * (total_pct / 100.0 if total_pct else 1.0)       # o que efetivamente vai às lojas

    # ---- meses entre o último dado e o início do ciclo: simulados com o % atual
    gap = (MES_INICIO_CICLO - (base["ultimo"].month + 1)) % 12
    S0 = base["S_util"].to_numpy(float)
    preso_cont = np.where(ativos, PESO_SALDO_PRESO * base["preso"].to_numpy(float), 0.0)
    if preso_cont.sum() > 0:                       # parte do saldo preso que se espera recuperar
        S0 = S0 + preso_cont
        logger.info("Etapa 4 | saldo preso contado a %.0f%% no rateio: %s kWh.",
                    PESO_SALDO_PRESO * 100, _r_fmt(preso_cont.sum()))
    if gap:
        og = _ciclo(meses12, base["ultimo"].month % 12 + 1)[:gap]
        S_gap, *_ = simular(p_atual, Uproj_cal[:, og], G_cal[og] * (total_pct / 100.0 if total_pct else 1.0), S0)
        S0 = S_gap[:, -1]
    logger.info("Etapa 4 | %s: ciclo começa em %s; %d mês(es) até a troca simulados com o %% atual.",
                entrada.get("sistema", ""), _rotulo_mes(MES_INICIO_CICLO), gap)

    # ---- saldo-alvo: colchão sazonal do mês de início + margem
    buf = buffer_sazonal(G_cal, Uproj_cal[ativos].sum(0) if ativos.any() else Uproj_cal.sum(0))
    buf_por_mes = {m.month: buf[i] for i, m in enumerate(meses12)}
    meses_buffer = buf_por_mes[MES_INICIO_CICLO][0] if USAR_BUFFER_SAZONAL else 0.0
    Uano = Uc.sum(1)
    alvo = Uano * (meses_buffer / 12 + MARGEM_SEGURANCA)

    # ---- métodos + otimização
    P, k = calcular_metodos(Uc, C12, G_benef.sum(), S0, base["S_total"].to_numpy(float),
                            pct_atual, ativos, alvo)
    if RODAR_OTIMIZACAO:
        p_opt = otimizar(Uc, G_benef, S0, P["Cobertura equalizada (W)"], ativos)
        if p_opt is not None:
            P["Otimização numérica"] = p_opt

    linhas = []
    for cen, (fg, fu) in CENARIOS.items():
        for nome, p in P.items():
            linhas.append(dict(cenario=cen, metodo=nome, **metricas(p, Uc * fu, G_benef * fg, S0, ativos)))
    comp_df = pd.DataFrame(linhas)

    # ---- % ideal final (W), arredondado conservando Σ% atual
    pW = P["Cobertura equalizada (W)"]
    pct_ideal = _r_arred({l: pW[i] * total_pct for i, l in enumerate(lojas)})
    pI = np.array([pct_ideal[l] for l in lojas]) / (total_pct if total_pct else 1.0)
    S_prop, comp_prop, fat_prop, rec_prop = simular(pI, Uc, G_benef, S0)
    S_atu, comp_atu, _, _ = simular(p_atual, Uc, G_benef, S0)

    # ---- sobra estrutural: geração + saldo acima do consumo (o saldo vai crescer)
    U_at = Uano[ativos].sum() if ativos.any() else 0.0
    sobra = G_benef.sum() + S0[ativos].sum() - U_at - alvo[ativos].sum()
    necessidade = np.where(ativos, np.maximum(0, np.minimum(k, 1.0) * Uano + alvo - S0), 0.0)
    cedivel = max(0.0, 1 - necessidade.sum() / G_benef.sum()) if G_benef.sum() else 0.0
    if sobra > 0.05 * max(U_at, 1):
        base["alertas"].append(dict(
            tipo="Excedente estrutural", loja="—", kwh=float(sobra),
            detalhe=f"geração do ciclo + saldo superam o consumo compensável — o saldo vai crescer. "
                    f"O saldo atual já cobre ~{S0[ativos].sum() / max(U_at / 12, 1):.0f} meses de consumo; "
                    f"≈ {cedivel * 100:.0f}% da energia do ciclo poderia ser rateada para outras UCs"))

    cd_m = base["cd_mensal"].reindex(lojas).fillna(CUSTO_DISP_PADRAO).to_numpy(float)
    obs = []
    for i, l in enumerate(lojas):
        o = []
        if l in excl:
            o.append("fora do rateio (sem compensação)" if l in base["sem_compensacao"] else "excluída (config)")
        if base["preso"][l] > 0:
            o.append(f"{_r_fmt(base['preso'][l])} kWh presos"
                     + (f" ({PESO_SALDO_PRESO:.0%} contado no rateio)" if ativos[i] and PESO_SALDO_PRESO else ""))
        if g[i] >= 1.10:
            o.append(f"consumo crescendo (+{(g[i] - 1) * 100:.0f}%)")
        if g[i] <= 0.92:
            o.append(f"consumo caindo ({(g[i] - 1) * 100:.0f}%)")
        if ativos[i] and Uano[i] <= 0:
            o.append("consumo coberto pela geração própria")
        elif ativos[i] and pW[i] == 0:
            o.append("saldo cobre o ciclo")
        obs.append("; ".join(o))
    tabela = pd.DataFrame({
        "ucs": base["ucs"].reindex(lojas).values,
        "consumo_anual_proj": Uc.sum(1) + cd_m * 12,
        "custo_disp_anual": cd_m * 12,
        "compensavel_proj": Uano,
        "saldo_atual": base["S_total"].values,
        "saldo_preso": base["preso"].values,
        "saldo_inicio": S0,
        "saldo_alvo": np.where(ativos, alvo, 0.0),
        "necessidade": np.where(ativos, np.maximum(0, k * Uano + alvo - S0) if np.isfinite(k) else 0.0, 0.0),
        "pct_atual": pct_atual,
        "pct_ideal": [pct_ideal[l] for l in lojas],
        "kwh_ideal_ano": pI * G_benef.sum(),
        "kwh_atual_ano": p_atual * G_benef.sum(),
        "saldo_min_proj": S_prop.min(1),
        "saldo_final_proj": S_prop[:, -1],
        "cobertura_proj": comp_prop.sum(1) / np.maximum(Uano, 1),
        "cobertura_atual": comp_atu.sum(1) / np.maximum(Uano, 1),
        "tendencia": g,
        "ativo": ativos,
        "observacao": obs,
    }, index=pd.Index(lojas, name="loja"))

    parametros = {
        "Sistema": entrada.get("sistema", ""),
        "Fonte": entrada.get("fonte", ""),
        "Estratégia": "Cobertura anual equalizada (water-filling)",
        "Início do ciclo": _rotulo_mes(MES_INICIO_CICLO),
        "Base histórica": f"{base['meses'][0]} a {base['ultimo']} ({len(base['meses'])} meses)",
        "Energia distribuível no ciclo (kWh)": Gano,
        "Σ % atual das beneficiárias": total_pct,
        "Energia às lojas no ciclo (kWh)": G_benef.sum(),
        "Consumo compensável projetado (kWh)": U_at,
        "Saldo utilizável no início (kWh)": S0[ativos].sum() if ativos.any() else 0.0,
        "Saldo preso (kWh)": base["preso"].sum(),
        "Cobertura comum k": k if np.isfinite(k) else 0.0,
        "Cobertura estrutural (G ÷ U)": G_benef.sum() / max(U_at, 1),
        "Parcela cedível a outras UCs": cedivel,
        "Colchão sazonal (meses)": meses_buffer,
        "Margem de segurança": MARGEM_SEGURANCA,
        "Peso do saldo preso": PESO_SALDO_PRESO,
        "Fator de geração": FATOR_GERACAO,
        "Amortecimento da tendência": TENDENCIA_AMORTECIMENTO,
        "Custo de disponibilidade": ", ".join(sorted({f"{v}" for v in base["origem_cd"].values()})),
    }
    bt = backtest_origem_movel(base, [l for l, a in zip(lojas, ativos) if a])
    logger.info("Etapa 4 | %s: k = %.1f%% | cobertura estrutural %.1f%% | Σ%% ideal = %.2f",
                entrada.get("sistema", ""), parametros["Cobertura comum k"] * 100,
                parametros["Cobertura estrutural (G ÷ U)"] * 100, sum(pct_ideal.values()))
    return dict(tabela=tabela, metodos=P, comparacao=comp_df, k=parametros["Cobertura comum k"],
                cobertura_estrutural=parametros["Cobertura estrutural (G ÷ U)"],
                parametros=parametros, alertas=base["alertas"], backtest=bt,
                meses_ciclo=[_rotulo_mes(MES_INICIO_CICLO + i) for i in range(12)],
                U_ciclo=Uc, G_ciclo=G_benef, ativos=ativos,
                S_prop=S_prop, S_atual=S_atu, rec_prop=rec_prop, fat_prop=fat_prop,
                buffer={m: buf_por_mes[m] for m in range(1, 13) if m in buf_por_mes},
                base=base, portal=entrada.get("portal", "site da distribuidora"))

## Etapa 5 — Saídas: relatório PDF por demonstrativo (e, opcionalmente, abas `Rateio *` no `.xlsx`)

**Excel**: por padrão (`GERAR_ABAS_RATEIO = False`) a planilha **não** recebe abas de rateio — se existirem de
execuções anteriores, são removidas. Com `GERAR_ABAS_RATEIO = True` as abas abaixo são recriadas a cada execução:

| Aba | Conteúdo |
|---|---|
| `Rateio Anual` | tabela final por loja. `k` (célula `B4`) é editável: **Necessidade**, **% recalculado** e **kWh/ano** são fórmulas. A coluna **% Ideal (digitar no site)** é o valor arredondado que soma exatamente o Σ% atual |
| `Rateio Simulação` | saldo projetado mês a mês de cada loja com o % Ideal e com o % Atual |
| `Rateio Métodos` | métricas de cada método × cenário de estresse, % de cada método e, se houver, o backtest |
| `Rateio Premissas` | parâmetros, premissas, hipóteses e alertas (saldo preso, UCs novas, sobra, etc.) |

**PDF** (um por demonstrativo): 1) painel executivo; 2) rateio proposto; 3) gráficos (sazonalidade,
saldo projetado, colchão por mês de início, cobertura por loja); 4) comparação de métodos; 5) premissas e alertas.

### 5.1 Abas `Rateio *` no Excel

In [ ]:
import openpyxl as _oxl
from openpyxl.styles import Alignment as _Al, Border as _Bo, Font as _Fo, PatternFill as _PF, Side as _Si
from openpyxl.utils import get_column_letter as _col

# mesmo padrão visual das planilhas PR/SC: cabeçalho cinza CCCCCC, texto preto em negrito, aba laranja
_R_HEADER, _R_HEADER2 = "000000", "CCCCCC"
_R_COR_ABA = "FF9900"
_R_FILL_H = _PF("solid", fgColor=_R_HEADER2)
_R_FILL_Z = _PF("solid", fgColor="F2F2F2")
_R_FONT_H = _Fo(bold=True, color="000000")
_R_BORDA = _Bo(bottom=_Si(style="thin", color="D9E2E4"))


def _r_cabecalho(ws, linha, titulos, larguras=None):
    for j, t in enumerate(titulos, 1):
        c = ws.cell(linha, j, t)
        c.fill, c.font = _R_FILL_H, _R_FONT_H
        c.alignment = _Al(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[linha].height = 32
    for j, w in enumerate(larguras or [], 1):
        ws.column_dimensions[_col(j)].width = w


def _r_titulo(ws, texto, sub=""):
    ws["A1"] = texto
    ws["A1"].font = _Fo(bold=True, size=13, color=_R_HEADER)
    ws["A1"].fill = _R_FILL_H
    ws.sheet_properties.tabColor = _R_COR_ABA
    if sub:
        ws["A2"] = sub
        ws["A2"].font = _Fo(italic=True, size=9, color="5A6B6B")


TEXTO_PREMISSAS = {
    "Dados observados": [
        "Consumo, créditos recebidos, % e saldo por UC vêm da aba Demonstrativo (extraída do PDF da distribuidora).",
        "O custo de disponibilidade (TR 100 / BI 50 / MO 30 kWh por mês) nunca é compensado: crédito usado = consumo − custo de disponibilidade.",
        "A geração é sazonal (verão > inverno) e o consumo das lojas costuma ser quase plano.",
    ],
    "Premissas": [
        "O % definido vale por 12 meses a partir do mês de início do ciclo e não é alterado no período.",
        "Todas as UCs pagam tarifa equivalente, então o custo total depende do saldo parado, não de qual UC recebe.",
        "Créditos valem 60 meses; saldo parado não se perde de imediato, mas é dinheiro imobilizado e risco de expirar.",
    ],
    "Hipóteses (validar)": [
        "Geração do próximo ciclo = últimos 12 meses × FATOR_GERACAO.",
        "Consumo do próximo ciclo = perfil dos últimos 12 meses × tendência amortecida por loja.",
        "O saldo preso será tratado com a distribuidora; até lá, não conta como saldo utilizável.",
    ],
    "Como usar": [
        "Digite no site da distribuidora o '% Ideal' da página 'Rateio anual proposto' deste relatório (soma = Σ % atual).",
        "UCs com 0% têm saldo suficiente para o ciclo, geração própria ou estão fora do rateio (ver Observação).",
        "Reavalie antes da próxima troca; com ≥ 24 meses de histórico o backtest roda automaticamente.",
    ],
}


def escrever_abas_rateio(rateio: dict, xlsx_path: Path) -> Path:
    """Recria as abas `Rateio *` na planilha informada."""
    xlsx_path = Path(xlsx_path)
    wb = _oxl.load_workbook(_io.BytesIO(xlsx_path.read_bytes()), data_only=False)
    removidas = [n for n in ABAS_RATEIO if n in wb.sheetnames]
    for nome in removidas:
        del wb[nome]
    if not GERAR_ABAS_RATEIO:                      # rateio só no relatório PDF
        if removidas:
            buf = _io.BytesIO(); wb.save(buf); xlsx_path.write_bytes(buf.getvalue())
            logger.info("Etapa 5 | abas de rateio removidas de %s: %s", xlsx_path.name, ", ".join(removidas))
        return xlsx_path
    tab, par = rateio["tabela"], rateio["parametros"]
    hoje = dt.date.today().strftime("%d/%m/%Y")

    # ------------------------------------------------------------ Rateio Anual
    ws = wb.create_sheet(ABAS_RATEIO[0])
    _r_titulo(ws, f"Rateio anual proposto — {par['Sistema']} — ciclo a partir de {par['Início do ciclo']}",
              f"Gerado em {hoje} · base {par['Base histórica']} · % fixo por 12 meses")
    ws["A4"], ws["B4"] = "Cobertura comum k", rateio["k"]
    ws["A5"], ws["B5"] = "Energia às UCs no ciclo (kWh)", par["Energia às lojas no ciclo (kWh)"]
    ws["A6"], ws["B6"] = "Σ % a distribuir", par["Σ % atual das beneficiárias"]
    ws["B4"].number_format, ws["B5"].number_format, ws["B6"].number_format = "0.0%", "#,##0", "0.00"
    ws["B4"].fill = _PF("solid", fgColor="FFF4D6")
    ws["C4"] = "← editável: altere k (ou use Atingir Meta para Σ Necessidade = B5) e veja as fórmulas"
    ws["C4"].font = _Fo(italic=True, size=9, color="5A6B6B")
    titulos = ["Loja / UC", "UC(s)", "Consumo anual proj. (kWh)", "Custo disp. anual (kWh)",
               "Compensável proj. (kWh)", "Saldo atual (kWh)", "Saldo preso (kWh)",
               "Saldo no início do ciclo (kWh)", "Saldo-alvo (kWh)", "Necessidade (kWh)",
               "% Atual", "% Ideal (digitar no site)", "% Ideal recalc. (fórmula)", "Δ p.p.",
               "kWh ideal / ano", "Saldo mínimo proj. (kWh)", "Saldo final proj. (kWh)",
               "Cobertura proj.", "Cobertura c/ % atual", "Tendência consumo", "Observação"]
    L0 = 8
    _r_cabecalho(ws, L0, titulos, [26, 22, 13, 11, 13, 11, 11, 13, 11, 13, 9, 12, 12, 8,
                                   12, 12, 12, 10, 10, 10, 46])
    tab_ord = tab.sort_values("pct_ideal", ascending=False)
    r1 = L0 + 1
    rN = r1 + len(tab_ord) - 1
    for i, (loja, t) in enumerate(tab_ord.iterrows()):
        r = r1 + i
        nec = f"=IF({'TRUE' if t['ativo'] else 'FALSE'},MAX(0,$B$4*E{r}+I{r}-H{r}),0)"
        vals = [loja, t["ucs"], t["consumo_anual_proj"], t["custo_disp_anual"], t["compensavel_proj"],
                t["saldo_atual"], t["saldo_preso"], t["saldo_inicio"], t["saldo_alvo"], nec,
                t["pct_atual"], t["pct_ideal"], f"=IFERROR(J{r}/SUM($J${r1}:$J${rN})*$B$6,0)",
                f"=L{r}-K{r}", f"=IFERROR(L{r}/$B$6*$B$5,0)", t["saldo_min_proj"], t["saldo_final_proj"],
                t["cobertura_proj"], t["cobertura_atual"], t["tendencia"] - 1, t["observacao"]]
        for j, v in enumerate(vals, 1):
            c = ws.cell(r, j, v.item() if hasattr(v, "item") else v)
            c.border = _R_BORDA
            if i % 2:
                c.fill = _R_FILL_Z
            if 3 <= j <= 10 or j in (15, 16, 17):
                c.number_format = "#,##0"
            elif j in (11, 12, 13, 14):
                c.number_format = "0.00"
            elif j in (18, 19):
                c.number_format = "0.0%"
            elif j == 20:
                c.number_format = "+0%;-0%;0%"
        ws.cell(r, 12).font = _Fo(bold=True)
    rt = rN + 1
    ws.cell(rt, 1, "TOTAL").font = _Fo(bold=True)
    for j in (3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15):
        L = _col(j)
        c = ws.cell(rt, j, f"=SUM({L}{r1}:{L}{rN})")
        c.font = _Fo(bold=True)
        c.number_format = "0.00" if j in (11, 12, 13) else "#,##0"
    ws.cell(rt + 1, 1, "Conferência: Σ % Ideal = Σ % a distribuir")
    ws.cell(rt + 1, 12, f'=IF(ROUND(L{rt}-$B$6,2)=0,"OK","DIFERENÇA")')
    ws.freeze_panes = ws.cell(r1, 2)

    # ------------------------------------------------------------ Rateio Simulação
    ws = wb.create_sheet(ABAS_RATEIO[1])
    _r_titulo(ws, "Saldo projetado mês a mês (kWh) — % fixo por 12 meses",
              "Hipótese: geração e consumo do ciclo repetem o perfil dos últimos 12 meses (com tendência).")
    lojas = list(tab.index)
    meses = rateio["meses_ciclo"]
    linha = 4
    for rotulo, S in [("Com % Ideal", rateio["S_prop"]), ("Com % Atual", rateio["S_atual"])]:
        ws.cell(linha, 1, rotulo).font = _Fo(bold=True, color=_R_HEADER)
        linha += 1
        _r_cabecalho(ws, linha, ["Loja / UC", "Início"] + meses, [26, 10] + [9] * 12)
        for i, l in enumerate(lojas):
            r = linha + 1 + i
            ws.cell(r, 1, l)
            ws.cell(r, 2, float(tab.loc[l, "saldo_inicio"])).number_format = "#,##0"
            for t in range(12):
                c = ws.cell(r, 3 + t, float(S[i, t]))
                c.number_format = "#,##0"
                if S[i, t] < 1 and rateio["U_ciclo"][i, t] > 0:
                    c.fill = _PF("solid", fgColor="FDECEC")
        r = linha + 1 + len(lojas)
        ws.cell(r, 1, "TOTAL").font = _Fo(bold=True)
        for j in range(2, 15):
            L = _col(j)
            c = ws.cell(r, j, f"=SUM({L}{linha + 1}:{L}{r - 1})")
            c.font, c.number_format = _Fo(bold=True), "#,##0"
        linha = r + 3
    ws.cell(linha, 1, "Células em vermelho: mês com saldo zerado (parte do consumo vai para a fatura).").font = \
        _Fo(italic=True, size=9, color="7A2E2E")

    # ------------------------------------------------------------ Rateio Métodos
    ws = wb.create_sheet(ABAS_RATEIO[2])
    _r_titulo(ws, "Comparação de métodos — % fixo por 12 meses sob cenários de estresse",
              "Menor é melhor em faturado, saldo ocioso e dispersão; maior é melhor em cobertura mínima.")
    comp = rateio["comparacao"]
    cols = [c for c in comp.columns if c not in ("cenario", "metodo")]
    _r_cabecalho(ws, 4, ["Cenário", "Método"] + cols, [26, 26] + [14] * len(cols))
    for i, (_, rr) in enumerate(comp.iterrows()):
        r = 5 + i
        ws.cell(r, 1, rr["cenario"]); ws.cell(r, 2, rr["metodo"])
        for j, c in enumerate(cols, 3):
            cel = ws.cell(r, j, float(rr[c]))
            cel.number_format = "0.0%" if "Cobertura" in c or "Dispersão" in c else (
                "0.0" if "meses" in c else "#,##0")
            if rr["metodo"] == "Cobertura equalizada (W)":
                cel.font = _Fo(bold=True)
    r = 5 + len(comp) + 2
    ws.cell(r, 1, "% por método (soma = Σ % a distribuir)").font = _Fo(bold=True, color=_R_HEADER)
    metodos = list(rateio["metodos"])
    _r_cabecalho(ws, r + 1, ["Loja / UC"] + metodos)
    tp = par["Σ % atual das beneficiárias"]
    for i, l in enumerate(lojas):
        ws.cell(r + 2 + i, 1, l)
        for j, m in enumerate(metodos, 2):
            ws.cell(r + 2 + i, j, float(rateio["metodos"][m][i] * tp)).number_format = "0.00"
    bt = rateio.get("backtest")
    r = r + 4 + len(lojas)
    ws.cell(r, 1, "Backtest com origem móvel").font = _Fo(bold=True, color=_R_HEADER)
    if bt is None or bt.empty:
        ws.cell(r + 1, 1, "Indisponível: requer ≥ 24 meses de histórico. Rode de novo quando o histórico completar 24 meses.")
    else:
        _r_cabecalho(ws, r + 1, list(bt.columns))
        for i, (_, rr) in enumerate(bt.iterrows()):
            for j, c in enumerate(bt.columns, 1):
                ws.cell(r + 2 + i, j, rr[c] if isinstance(rr[c], str) else float(rr[c]))

    # ------------------------------------------------------------ Rateio Premissas
    ws = wb.create_sheet(ABAS_RATEIO[3])
    _r_titulo(ws, f"Premissas, parâmetros e alertas do rateio anual — {par['Sistema']}")
    _r_cabecalho(ws, 3, ["Parâmetro", "Valor"], [44, 70])
    for i, (kk, v) in enumerate(par.items()):
        ws.cell(4 + i, 1, kk)
        c = ws.cell(4 + i, 2, v.item() if hasattr(v, "item") else v)
        if isinstance(v, float):
            c.number_format = ("0.0%" if kk.startswith(("Cobertura", "Margem", "Parcela"))
                               else ("0.00" if abs(v) < 50 else "#,##0"))
    r = 5 + len(par)
    for bloco, itens in TEXTO_PREMISSAS.items():
        ws.cell(r, 1, bloco).font = _Fo(bold=True, color=_R_HEADER)
        for it in itens:
            r += 1
            ws.cell(r, 1, "• " + it).alignment = _Al(wrap_text=True, vertical="top")
            ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
            ws.row_dimensions[r].height = 30
        r += 2
    ws.cell(r, 1, "Alertas").font = _Fo(bold=True, color="B23B3B")
    _r_cabecalho(ws, r + 1, ["Tipo / Loja", "Detalhe"])
    for i, a in enumerate(rateio["alertas"]):
        ws.cell(r + 2 + i, 1, f"{a['tipo']} — {a['loja']}")
        kwh = f" ({_r_fmt(a['kwh'])} kWh)" if a.get("kwh") else ""
        ws.cell(r + 2 + i, 2, a["detalhe"] + kwh).alignment = _Al(wrap_text=True)

    buf = _io.BytesIO()
    wb.save(buf)
    xlsx_path.write_bytes(buf.getvalue())
    logger.info("Etapa 5 | abas %s gravadas em %s", ", ".join(ABAS_RATEIO), xlsx_path.name)
    return xlsx_path

### 5.2 Relatório PDF (painel + rateio + gráficos + métodos + premissas)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.patches import FancyBboxPatch
from matplotlib.backends.backend_pdf import PdfPages

R_FUNDO, R_HEADER, R_HEADER2, R_TXT = "#f4f6f8", "#0f3d3e", "#155e63", "#1c2b2b"
R_LINHA_A, R_LINHA_B = "#ffffff", "#eef2f3"
COR_PROP, COR_ATUAL = "#155e63", "#c98a1a"      # séries "% Ideal" e "% Atual"
COR_NEUTRA, COR_GRADE = "#9aa7a7", "#d9e2e4"


def _pagina_base(titulo: str, subtitulo: str, altura: float = 11.0):
    fig = plt.figure(figsize=(13.6, altura), dpi=140)
    fig.patch.set_facecolor(R_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    H = altura
    ax.add_patch(FancyBboxPatch((0.03, 1 - 1.0 / H), 0.94, 0.85 / H,
                 boxstyle="round,pad=0.002,rounding_size=0.006", fc=R_HEADER, ec="none", zorder=2))
    ax.text(0.05, 1 - 0.43 / H, titulo, fontsize=17, fontweight="bold", color="white", va="center", zorder=3)
    ax.text(0.05, 1 - 0.78 / H, subtitulo, fontsize=9.5, color="#cfe3e3", va="center", zorder=3)
    return fig, ax, H


def _estilo_eixo_r(a):
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        a.spines[s].set_color(COR_GRADE)
    a.tick_params(colors="#5a6b6b", labelsize=8)
    a.grid(axis="y", color=COR_GRADE, lw=0.6)
    a.set_axisbelow(True)
    a.set_facecolor("white")


def _quebrar(texto: str, largura: int) -> list:
    palavras, linhas, atual = texto.split(), [], ""
    for p in palavras:
        if len(atual) + len(p) + 1 > largura:
            linhas.append(atual); atual = ""
        atual += (" " if atual else "") + p
    return linhas + ([atual] if atual else [])


def pagina_painel_png(png: Path):
    """Página 1 a partir do PNG do painel já gerado pelo notebook."""
    img = plt.imread(str(png))
    h, w = img.shape[:2]
    fig = plt.figure(figsize=(13.6, 13.6 * h / w), dpi=140)
    fig.patch.set_facecolor(R_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.imshow(img)
    return fig


def pagina_tabela_rateio(rateio: dict):
    tab = rateio["tabela"].sort_values("pct_ideal", ascending=False)
    par = rateio["parametros"]
    n = len(tab)
    row_h = 0.26
    H = 3.4 + max(n, 4) * row_h
    fig, ax, H = _pagina_base(
        f"Rateio anual proposto — {par['Sistema']} — ciclo a partir de {par['Início do ciclo']}",
        f"Cobertura comum k = {rateio['k'] * 100:.1f}%  ·  energia às UCs no ciclo ≈ "
        f"{_r_fmt(par['Energia às lojas no ciclo (kWh)'])} kWh  ·  Σ% = {par['Σ % atual das beneficiárias']:.2f}"
        f"  ·  % fixo por 12 meses", H)
    def y(px): return 1 - px / H
    X = dict(comp=0.30, util=0.365, preso=0.43, atual=0.485, ideal=0.545, dk=0.62, smin=0.685,
             sfin=0.75, cob=0.835, obs=0.845)
    cols = [("Loja / UC", 0.045, "left"), ("Compensável\nproj. (kWh/ano)", X["comp"], "right"),
            ("Saldo útil\n(kWh)", X["util"], "right"), ("Saldo preso\n(kWh)", X["preso"], "right"),
            ("% Atual", X["atual"], "right"), ("% Ideal", X["ideal"], "right"), ("Δ kWh\n/ano", X["dk"], "right"),
            ("Saldo mín.\nproj.", X["smin"], "right"), ("Saldo final\nproj.", X["sfin"], "right"),
            ("Cobertura\natual → proj.", X["cob"], "right"), ("Observação", X["obs"], "left")]
    yh = 1.25
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.5)), 0.94, 0.5 / H, fc=R_HEADER2, ec="none", zorder=2))
    for nome, cx, al in cols:
        ax.text(cx, y(yh + 0.25), nome, fontsize=7.8, fontweight="bold", color="white",
                ha=al, va="center", zorder=3, linespacing=0.9)
    y0 = yh + 0.5
    for i, (loja, t) in enumerate(tab.iterrows()):
        yy = y0 + i * row_h
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc=R_LINHA_A if i % 2 == 0 else R_LINHA_B, ec="none", zorder=1))
        yc = y(yy + row_h / 2)
        kw = dict(fontsize=8, va="center", zorder=3, color=R_TXT if t["ativo"] else "#9aa7a7")
        ax.text(0.045, yc, str(loja)[:40], ha="left", **kw)
        ax.text(X["comp"], yc, _r_fmt(t["compensavel_proj"]), ha="right", **kw)
        ax.text(X["util"], yc, _r_fmt(t["saldo_inicio"]), ha="right", **kw)
        ax.text(X["preso"], yc, _r_fmt(t["saldo_preso"]) if t["saldo_preso"] else "—", ha="right",
                **{**kw, "color": "#b23b3b" if t["saldo_preso"] else "#9aa7a7"})
        ax.text(X["atual"], yc, f"{t['pct_atual']:.2f}", ha="right", **kw)
        d = t["pct_ideal"] - t["pct_atual"]
        ax.text(X["ideal"], yc, f"{t['pct_ideal']:.2f}", ha="right", fontweight="bold", **kw)
        seta, cs = ("▲", "#1a7f5a") if d > 0.05 else (("▼", "#b23b3b") if d < -0.05 else ("=", "#5a6b6b"))
        ax.text(X["ideal"] + 0.005, yc, seta, fontsize=6.4, ha="left", va="center", color=cs, zorder=3)
        dk = t["kwh_ideal_ano"] - t["kwh_atual_ano"]
        ax.text(X["dk"], yc, _r_fmt_sinal(dk), ha="right",
                **{**kw, "color": "#1a7f5a" if dk > 0 else ("#b23b3b" if dk < 0 else "#5a6b6b")})
        ax.text(X["smin"], yc, _r_fmt(t["saldo_min_proj"]), ha="right", **kw)
        ax.text(X["sfin"], yc, _r_fmt(t["saldo_final_proj"]), ha="right", **kw)
        cob_a, cob_p = min(t["cobertura_atual"], 9.99), min(t["cobertura_proj"], 9.99)
        ax.text(X["cob"], yc, f"{cob_a * 100:.0f}% → {cob_p * 100:.0f}%", ha="right", **kw)
        ax.text(X["obs"], yc, (t["observacao"][:30] + "…") if len(t["observacao"]) > 31 else t["observacao"],
                ha="left", fontsize=6.2, va="center", color="#7a2e2e", zorder=3)
    fy = y0 + max(n, 4) * row_h + 0.3
    ax.text(0.03, y(fy),
            "Compensável = consumo − custo de disponibilidade − geração própria.  Saldo útil = saldo no início do ciclo "
            "sem o saldo preso.  Em cinza: fora do rateio.  Δ kWh/ano = energia recebida no ciclo com % Ideal − com % Atual.",
            fontsize=7, color="#8a9797", va="center")
    return fig


def pagina_graficos(rateio: dict):
    fig, ax0, H = _pagina_base("Por que o % precisa aguentar o ano inteiro",
                               f"{rateio['parametros']['Sistema']} · sazonalidade da geração, trajetória do saldo com % fixo, "
                               "colchão por mês de início e cobertura anual por UC", 11.5)
    gs = fig.add_gridspec(3, 2, left=0.07, right=0.97, top=1 - 1.35 / H, bottom=0.06,
                          hspace=0.55, wspace=0.28, width_ratios=[1.25, 1])
    meses = rateio["meses_ciclo"]; x = np.arange(12)
    at = rateio["ativos"] if rateio["ativos"].any() else np.ones_like(rateio["ativos"])
    a = fig.add_subplot(gs[0, 0]); _estilo_eixo_r(a)
    G = rateio["G_ciclo"]; U = rateio["U_ciclo"][at].sum(0)
    a.plot(x, G / max(G.sum(), 1) * 100, color=COR_PROP, lw=2, marker="o", ms=4, label="Energia distribuída")
    if U.sum() > 0:
        a.plot(x, U / U.sum() * 100, color=COR_ATUAL, lw=2, marker="o", ms=4, label="Consumo compensável")
    a.axhline(100 / 12, color=COR_NEUTRA, lw=0.8, ls="--")
    a.set_xticks(x, meses); a.set_ylabel("% do total anual", fontsize=8, color="#5a6b6b")
    a.set_title("Geração × consumo ao longo do ciclo", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    a.legend(frameon=False, fontsize=7.5, loc="upper right")
    b = fig.add_subplot(gs[1, 0]); _estilo_eixo_r(b)
    b.plot(x, rateio["S_atual"][at].sum(0), color=COR_ATUAL, lw=2, marker="o", ms=4, label="% Atual")
    b.plot(x, rateio["S_prop"][at].sum(0), color=COR_PROP, lw=2, marker="o", ms=4, label="% Ideal")
    b.set_xticks(x, meses); b.set_ylabel("kWh", fontsize=8, color="#5a6b6b")
    b.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: _r_fmt(v)))
    b.set_title("Saldo total projetado (% fixo)", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    b.legend(frameon=False, fontsize=7.5)
    c = fig.add_subplot(gs[2, 0]); _estilo_eixo_r(c)
    buf = rateio["buffer"]
    vals = [buf.get(m, (0, 0))[0] for m in range(1, 13)]
    cores = [COR_PROP if m == MES_INICIO_CICLO else "#b9c6c6" for m in range(1, 13)]
    c.bar(np.arange(12), vals, color=cores, width=0.7, edgecolor=R_FUNDO, lw=1)
    c.set_xticks(np.arange(12), [_rotulo_mes(m) for m in range(1, 13)])
    c.set_ylabel("meses de consumo", fontsize=8, color="#5a6b6b")
    c.set_title(f"Colchão de saldo exigido por mês de troca (atual: {_rotulo_mes(MES_INICIO_CICLO)})",
                fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    for i, v in enumerate(vals):
        c.text(i, v + 0.03, f"{v:.1f}", ha="center", fontsize=7, color="#5a6b6b")
    i_esc = MES_INICIO_CICLO - 1
    c.annotate("mês da troca", (i_esc, vals[i_esc]), xytext=(i_esc, max(max(vals), 0.2) * 0.55), ha="center",
               fontsize=7.5, color=COR_PROP, fontweight="bold",
               arrowprops=dict(arrowstyle="-|>", color=COR_PROP, lw=1))
    d = fig.add_subplot(gs[:, 1]); _estilo_eixo_r(d); d.grid(axis="y", visible=False)
    d.grid(axis="x", color=COR_GRADE, lw=0.6)
    tab = rateio["tabela"][rateio["tabela"]["ativo"] & (rateio["tabela"]["compensavel_proj"] > 0)]
    tab = tab.sort_values("cobertura_atual", ascending=True)       # pior cobertura atual no topo
    n_lin = max(len(tab), 8)
    yy = n_lin - 1 - np.arange(len(tab))
    ca = np.minimum(tab["cobertura_atual"].to_numpy() * 100, 100)
    cp = np.minimum(tab["cobertura_proj"].to_numpy() * 100, 100)
    d.hlines(yy, np.minimum(ca, cp), np.maximum(ca, cp), color=COR_GRADE, lw=2)
    d.scatter(ca, yy, color=COR_ATUAL, s=26, zorder=3, label="% Atual", edgecolor="white", lw=0.8)
    d.scatter(cp, yy, color=COR_PROP, s=26, zorder=4, label="% Ideal", edgecolor="white", lw=0.8)
    d.set_yticks(yy, [str(i)[:30] for i in tab.index], fontsize=6.8 if len(tab) > 12 else 8)
    d.set_ylim(-0.6, n_lin - 0.4)
    d.set_xlabel("cobertura do consumo compensável no ciclo (%)", fontsize=8, color="#5a6b6b")
    d.set_title("Cobertura anual por UC", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    d.legend(frameon=False, fontsize=7.5, loc="lower right")
    return fig


def pagina_metodos(rateio: dict):
    comp = rateio["comparacao"]
    base = comp[comp["cenario"] == "Base"].set_index("metodo")
    pior = comp.groupby("metodo").agg({"Faturado acima do mínimo (kWh)": "max",
                                       "Saldo ocioso >1 mês (kWh)": "max", "Cobertura mín.": "min"})
    metodos = list(base.index)
    fig, ax, H = _pagina_base(f"Comparação de métodos — {rateio['parametros']['Sistema']} — % fixo por 12 meses",
                              "Cenário base (perfil dos últimos 12 meses) e pior caso entre os cenários de estresse: "
                              + ", ".join(CENARIOS), 8.5)
    def y(px): return 1 - px / H
    cols = [("Método", 0.045, "left"), ("Faturado\n(kWh)", 0.31, "right"),
            ("Faturado acima\ndo mínimo", 0.40, "right"), ("Saldo final\n(kWh)", 0.48, "right"),
            ("Saldo ocioso\n>1 mês", 0.56, "right"), ("Cobertura\nmínima", 0.63, "right"),
            ("Dispersão\ncobertura", 0.70, "right"),
            ("PIOR CASO\nfat. acima mín.", 0.80, "right"), ("PIOR CASO\nsaldo ocioso", 0.88, "right"),
            ("PIOR CASO\ncob. mínima", 0.955, "right")]
    yh = 1.3
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.55)), 0.94, 0.55 / H, fc=R_HEADER2, ec="none", zorder=2))
    for nome, cx, al in cols:
        ax.text(cx, y(yh + 0.27), nome, fontsize=7.8, fontweight="bold", color="white", ha=al, va="center",
                zorder=3, linespacing=0.9)
    row_h = 0.42
    for i, m in enumerate(metodos):
        yy = yh + 0.55 + i * row_h
        destaque = m == "Cobertura equalizada (W)"
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc="#e9f6ef" if destaque else (R_LINHA_A if i % 2 == 0 else R_LINHA_B), ec="none", zorder=1))
        yc = y(yy + row_h / 2); bb = base.loc[m]; p = pior.loc[m]
        kw = dict(fontsize=8.6, va="center", zorder=3, color=R_TXT, fontweight="bold" if destaque else "normal")
        vals = [m + ("  ← recomendado" if destaque else ""), _r_fmt(bb["Faturado (kWh)"]),
                _r_fmt(bb["Faturado acima do mínimo (kWh)"]), _r_fmt(bb["Saldo final total (kWh)"]),
                _r_fmt(bb["Saldo ocioso >1 mês (kWh)"]), f"{bb['Cobertura mín.'] * 100:.0f}%",
                f"{bb['Dispersão cobertura (dp)'] * 100:.1f} p.p.",
                _r_fmt(p["Faturado acima do mínimo (kWh)"]), _r_fmt(p["Saldo ocioso >1 mês (kWh)"]),
                f"{p['Cobertura mín.'] * 100:.0f}%"]
        for (nome, cx, al), v in zip(cols, vals):
            ax.text(cx, yc, v, ha=al, **kw)
    fy = yh + 0.55 + len(metodos) * row_h + 0.45
    notas = [
        "Faturado acima do mínimo = energia paga na fatura além do déficit inevitável (consumo − geração − saldo). É o que um rateio melhor pode evitar.",
        "Saldo ocioso = saldo final acima de 1 mês de consumo: crédito parado numa UC enquanto outras pagam fatura.",
        "Cobertura mínima / dispersão = justiça entre UCs. Numa mesma empresa o custo total depende do saldo parado, não de qual UC recebe.",
        "Com poucas UCs ou geração sobrando, os métodos ficam próximos; a diferença aparece no saldo ocioso.",
    ]
    bt = rateio.get("backtest")
    notas.append("Backtest com origem móvel: " + ("indisponível (requer ≥ 24 meses de histórico)."
                                                 if bt is None or bt.empty else
                                                 f"{bt['origem'].nunique()} origens avaliadas — ver aba 'Rateio Métodos'."))
    for i, t in enumerate(notas):
        ax.text(0.04, y(fy + i * 0.32), "• " + t, fontsize=8.2, color="#3c4b4b", va="center")
    return fig


def pagina_premissas(rateio: dict):
    fig, ax, H = _pagina_base(f"Premissas, parâmetros e alertas — {rateio['parametros']['Sistema']}",
                              f"Separação entre dado observado, premissa e hipótese — revise antes de aplicar no {rateio['portal']}",
                              11.0)
    def y(px): return 1 - px / H
    yy = 1.35
    ax.text(0.04, y(yy), "Parâmetros do cálculo", fontsize=11, fontweight="bold", color=R_HEADER2)
    yy += 0.3
    for kk, v in rateio["parametros"].items():
        if isinstance(v, float):
            v = (f"{v * 100:.1f}%" if kk.startswith(("Cobertura", "Margem", "Parcela"))
                 else (f"{v:.2f}" if abs(v) < 50 else f"{_r_fmt(v)}"))
        ax.text(0.05, y(yy), kk, fontsize=8.2, color="#5a6b6b", va="center")
        ax.text(0.28, y(yy), str(v)[:48], fontsize=8.2, color=R_TXT, va="center", fontweight="bold")
        yy += 0.21
    yy2 = 1.35
    ax.text(0.52, y(yy2), "Alertas", fontsize=11, fontweight="bold", color="#b23b3b")
    yy2 += 0.3
    if not rateio["alertas"]:
        ax.text(0.53, y(yy2), "Nenhum alerta.", fontsize=8, color="#5a6b6b", va="center")
    for a in rateio["alertas"][:22]:
        kwh = f" ({_r_fmt(a['kwh'])} kWh)" if a.get("kwh") else ""
        txt = f"{a['tipo']} — {a['loja']}{kwh}: {a['detalhe']}"
        for j, linha in enumerate(_quebrar(txt, 92)):
            ax.text(0.53 if j == 0 else 0.545, y(yy2), ("⚠ " if j == 0 else "") + linha,
                    fontsize=7.6, color="#7a2e2e", va="center")
            yy2 += 0.19
        yy2 += 0.05
    yy = max(yy, yy2) + 0.3
    for bloco, itens in TEXTO_PREMISSAS.items():
        ax.text(0.04, y(yy), bloco, fontsize=10, fontweight="bold", color=R_HEADER2)
        yy += 0.27
        for it in itens:
            for j, linha in enumerate(_quebrar(it, 170)):
                ax.text(0.05, y(yy), ("• " if j == 0 else "   ") + linha, fontsize=8, color="#3c4b4b", va="center")
                yy += 0.2
        yy += 0.12
    ax.text(0.03, y(H - 0.25), f"Gerado automaticamente em {dt.date.today().strftime('%d/%m/%Y')}  ·  "
            f"Fonte: {rateio['parametros']['Fonte']}", fontsize=7, color="#8a9797", va="center")
    return fig


def gerar_relatorio_pdf(rateio: dict, painel=None, saida: Path | None = None) -> Path:
    """Relatório completo de UM demonstrativo em um único PDF.
    `painel`: função que devolve a figura do painel, caminho do PNG do painel, ou None."""
    paginas = []
    if callable(painel):
        paginas.append(painel)
    elif painel is not None and Path(painel).exists():
        paginas.append(lambda: pagina_painel_png(Path(painel)))
    paginas += [lambda: pagina_tabela_rateio(rateio), lambda: pagina_graficos(rateio),
                lambda: pagina_metodos(rateio), lambda: pagina_premissas(rateio)]
    with PdfPages(str(saida)) as pdf:
        for fazer in paginas:
            fig = fazer()
            pdf.savefig(fig, facecolor=R_FUNDO)
            plt.close(fig)
        info = pdf.infodict()
        info["Title"] = f"Energia Injetada e Rateio Anual — {rateio['parametros']['Sistema']}"
    logger.info("Etapa 5 | relatório PDF salvo em: %s", Path(saida).name)
    return Path(saida)

## 6. Pipeline completo — `main()`
Cada etapa é independente: se o PDF falhar, a planilha é só reconciliada e o painel, o rateio e o relatório saem com os dados já gravados.

In [ ]:
def main(pdf_path: Path = PDF_PATH,
         xlsx_in: Path = XLSX_PATH,
         xlsx_out: Path = XLSX_OUT,
         painel_out: Path = PAINEL_OUT,
         relatorio_out: Path = RELATORIO_OUT,
         apenas_ultimo_mes: bool = APENAS_ULTIMO_MES):
    """Pipeline: 1 (PDF -> DataFrame) -> 2 (planilha SC) -> 4 (rateio anual) -> 3 (painel) -> 5 (xlsx + PDF)."""
    logger.info("=" * 68)
    logger.info("INÍCIO DO PIPELINE — Energia Injetada (Celesc / SC) v2")
    logger.info("=" * 68)

    df = None
    validacao = pd.DataFrame()
    xlsx_para_painel = xlsx_in       # se a Etapa 2 falhar, o painel usa a planilha original

    # -------- Etapa 1: extração do PDF --------
    try:
        df = extrair_pdf(pdf_path, apenas_ultimo_mes=apenas_ultimo_mes)
        if df is None or df.empty:
            logger.warning("Etapa 1 | nenhuma linha extraída do PDF.")
            df = None
        else:
            logger.info("Etapa 1 | %d linha(s) extraída(s) do PDF.", len(df))
            validacao = validar_extracao(df)
    except Exception as e:
        logger.error("Etapa 1 | falha na extração do PDF: %s", e)
        logger.warning("Etapa 1 | seguindo sem novos dados (o painel usará a planilha atual).")
        df = None

    # -------- Etapa 2: planilha SC (cria se necessário; df=None -> só reconciliação) --------
    try:
        xlsx_out = atualizar_excel(df, xlsx_in, xlsx_out)
        xlsx_para_painel = xlsx_out
        logger.info("Etapa 2 | planilha atualizada com sucesso.")
    except Exception as e:
        logger.exception("Etapa 2 | falha ao atualizar a planilha: %s", e)
        xlsx_para_painel = xlsx_in

    # -------- Etapa 4: rateio anual (antes do painel, que usa a % Ideal) --------
    rateio = None
    try:
        rateio = calcular_rateio_anual(entrada_rateio_celesc(xlsx_para_painel))
        logger.info("Etapa 4 | rateio anual calculado.")
    except Exception as e:
        logger.exception("Etapa 4 | falha no rateio anual: %s", e)

    # -------- Etapa 3: painel visual --------
    # (v2) se o último mês ainda não tem os créditos lançados, o painel usa o último mês completo
    fixado = False
    if rateio is not None and MES_REFERENCIA_PAINEL is None and any(
            a["tipo"] == "Mês incompleto" for a in rateio["alertas"]):
        globals()["MES_REFERENCIA_PAINEL"] = str(rateio["base"]["ultimo"])
        fixado = True
        logger.info("Etapa 3 | painel fixado em %s (último mês com créditos lançados).", MES_REFERENCIA_PAINEL)
    dados = maxmes = png = pdf = None
    try:
        dados, maxmes, png, pdf = gerar_relatorio_visual(xlsx_para_painel, painel_out, rateio)
        logger.info("Etapa 3 | painel gerado com sucesso.")
    except Exception as e:
        logger.exception("Etapa 3 | falha ao gerar o painel: %s", e)
    finally:
        if fixado:
            globals()["MES_REFERENCIA_PAINEL"] = None

    # -------- Etapa 5: abas Rateio no xlsx + relatório PDF --------
    relatorio = None
    if rateio is not None:
        try:
            escrever_abas_rateio(rateio, xlsx_para_painel)
        except Exception as e:
            logger.exception("Etapa 5 | falha ao gravar abas do rateio: %s", e)
        try:
            relatorio = gerar_relatorio_pdf(rateio, png, relatorio_out)
        except Exception as e:
            logger.exception("Etapa 5 | falha ao gerar o relatório PDF: %s", e)

    logger.info("=" * 68)
    logger.info("FIM DO PIPELINE")
    for rot, p in [("Planilha", xlsx_para_painel), ("Painel", png), ("Relatório", relatorio)]:
        if p:
            logger.info("%s: %s", rot, p)
    logger.info("=" * 68)
    return dict(df=df, validacao=validacao, dados=dados, mes=maxmes, xlsx=xlsx_para_painel, png=png,
                pdf=pdf, rateio=rateio, relatorio=relatorio)


# Execute o pipeline completo:
resultado = main()

### 5.1 Validação da extração e pré-visualização do painel

In [ ]:
from IPython.display import Image, display

val = resultado.get("validacao")
if val is not None and not val.empty:
    print("Ocorrências da validação (conferir contra o PDF):")
    display(val)
else:
    print("Validação: nenhuma divergência.")

if resultado.get("df") is not None:
    print("\nÚltimo mês por UC extraído do PDF:")
    display(resultado["df"].sort_values("data_referencia").groupby("uc_conces").tail(1)
            [["uc_conces", "tipo", "data_referencia", "consumo", "cred_receb_fp", "percentual", "saldo_final_fp"]])

if resultado.get("png") and Path(resultado["png"]).exists():
    display(Image(filename=str(resultado["png"])))

if resultado.get("rateio"):
    t = resultado["rateio"]["tabela"]
    display(t[["compensavel_proj", "saldo_inicio", "saldo_preso", "pct_atual", "pct_ideal",
               "saldo_final_proj", "cobertura_proj", "observacao"]].sort_values("pct_ideal", ascending=False))

## 6. Notas de operação (específicas da Celesc)

**Rotina mensal.** Baixe o novo "Relatório do Saldo de Energia" no portal da Celesc, salve como
`RelatorioSaldoCelesc.pdf` e rode o notebook. Na primeira execução a planilha SC é criada do zero; nas seguintes
ela é carregada (de `XLSX_PATH`, ou da última saída em `saida/` se `XLSX_PATH` não existir) e atualizada.
Como cada relatório traz ~13 meses, rodar todo mês mantém o histórico completo e corrige meses revisados.

**Mês mais recente incompleto.** No relatório de exemplo, em 09/2026 a geradora já distribuiu 1.722 kWh, mas as
beneficiárias aparecem com *Energia injetada = 0* — o crédito ainda não havia sido lançado. A validação marca isso
como "mês mais recente: crédito ainda não lançado". Com `SOBRESCREVER_MESES=True`, o relatório do mês seguinte corrige a linha
automaticamente. Até lá, o painel desse mês tende a mostrar as lojas com saldo menor do que o real.

**Percentual.** Vem do rodapé e representa a **participação vigente na emissão** do relatório. Como é aplicado a
todos os meses daquela UC, meses antigos podem mostrar a % atual, e não a da época (a validação "rateio" acusa
quando a % mudou). A geradora recebe `percentual = 0` (ela lista a si mesma com 100%/0%, o que não é crédito recebido).

**Geradora.** Identificada pela coluna *Excedentes distribuídos*. Na geradora, `cred_receb_fp` guarda a energia
injetada bruta; ela é excluída dos KPIs e do rateio pelo `TAG_GERADORA` no nome — mantenha "(GERADORA)" no
`MAPA_UC_LOJA`.

**Parser de números.** `limpar_numero()` desta versão trata `ponto + 3 dígitos` como milhar (`2.479` = 2479).
Não misture com a função da Copel, onde `2.32` é decimal.

**Parâmetros da % Ideal** (célula de CONFIG): `HORIZONTE_MESES`, `CONSIDERAR_SALDO`, `JANELA_MEDIA_MESES`,
`MESES_BUFFER`, `PISO_PCT` — mesma lógica da versão Copel.

**Idempotência / reparo.** Chave `(UC, AAAA-MM)`. Rodar duas vezes o mesmo PDF não duplica. `reparar_planilha()`
recria meses apagados das abas de loja sem ler PDF.

**(v2) Rateio anual e relatório.** Parâmetros na célula 4.0 (iguais aos do notebook Copel). Saídas: abas `Rateio *` na planilha SC e `saida/relatorio_energia_injetada_sc.pdf`. Como o relatório Celesc não traz o custo de disponibilidade, o rateio usa `CUSTO_DISP_PADRAO` (100 kWh) — ajuste por UC em `CUSTO_DISP_MANUAL` se alguma for bifásica/monofásica.